# Brazil E-Commerce Analysis


## 1. Project Overview


### Project Scope

This project examines a Brazilian e-commerce marketplace through sales, customer, product, seller, payment, delivery, review, and regional performance data. It connects these perspectives to provide a structured view of commercial activity and the end-to-end customer journey.


### Project Motivation

The project was created to demonstrate how marketplace transaction data can be transformed into actionable business insight. Its purpose is to identify opportunities related to growth, operational efficiency, and customer experience through a practical e-commerce analytics workflow.


## 2. Business Objectives


This project analyzes macro-level Brazilian e-commerce performance and marketplace trends. It examines market share proxies, brand and product categories, company competitiveness, and customer purchasing power to identify the commercial and operational factors that shape marketplace performance.

The analysis focuses on order volume, revenue, delivery, customer behavior, seller performance, category performance, payment patterns, and customer reviews. Market share and competitive position are evaluated within the available marketplace data using seller- and category-level measures.


## 3. Dataset Overview


### 3.1 Data Sources

The analysis uses nine CSV files from the Olist Brazilian E-Commerce dataset. Together, they describe the order lifecycle, customer and seller locations, products, payments, reviews, and geographic reference data.

| Source File                             | Grain                                    | Main Purpose                                                      |
| --------------------------------------- | ---------------------------------------- | ----------------------------------------------------------------- |
| `olist_orders_dataset.csv`              | One row per order                        | Order status and order lifecycle timestamps.                      |
| `olist_order_items_dataset.csv`         | One row per item within an order         | Item price, freight charge, product, and seller information.      |
| `olist_order_payments_dataset.csv`      | One row per payment record               | Payment method, installment count, and payment value.             |
| `olist_order_reviews_dataset.csv`       | One row per customer review              | Customer rating, review text, and response timestamps.            |
| `olist_customers_dataset.csv`           | One row per customer-order record        | Customer identity and location attributes.                        |
| `olist_products_dataset.csv`            | One row per product                      | Product category and physical product attributes.                 |
| `olist_sellers_dataset.csv`             | One row per seller                       | Seller identity and location attributes.                          |
| `olist_geolocation_dataset.csv`         | One row per geographic coordinate record | Postal-code, city, state, latitude, and longitude reference data. |
| `product_category_name_translation.csv` | One row per product category translation | Portuguese-to-English category mapping.                           |


### 3.2 Data Model and Relationships

The `olist_orders_dataset.csv` file is the central transaction table. It connects each order to the customer, item, payment, and review records needed for marketplace analysis.

| From File                          | Join Key                   | To File                                 | Relationship                                                       |
| ---------------------------------- | -------------------------- | --------------------------------------- | ------------------------------------------------------------------ |
| `olist_orders_dataset.csv`         | `customer_id`              | `olist_customers_dataset.csv`           | Each order links to one customer-order record.                     |
| `olist_order_items_dataset.csv`    | `order_id`                 | `olist_orders_dataset.csv`              | An order can contain multiple item records.                        |
| `olist_order_payments_dataset.csv` | `order_id`                 | `olist_orders_dataset.csv`              | An order can have one or more payment records.                     |
| `olist_order_reviews_dataset.csv`  | `order_id`                 | `olist_orders_dataset.csv`              | An order can be linked to one or more review records.              |
| `olist_order_items_dataset.csv`    | `product_id`               | `olist_products_dataset.csv`            | Each item links to one product.                                    |
| `olist_order_items_dataset.csv`    | `seller_id`                | `olist_sellers_dataset.csv`             | Each item links to one seller.                                     |
| `olist_products_dataset.csv`       | `product_category_name`    | `product_category_name_translation.csv` | Product categories can be translated from Portuguese to English.   |
| `olist_customers_dataset.csv`      | `customer_zip_code_prefix` | `olist_geolocation_dataset.csv`         | Customer locations can be enriched with geographic reference data. |
| `olist_sellers_dataset.csv`        | `seller_zip_code_prefix`   | `olist_geolocation_dataset.csv`         | Seller locations can be enriched with geographic reference data.   |

The geolocation file may contain multiple coordinate records for the same postal-code prefix. Geographic analysis should therefore aggregate this file to one representative location per postal-code prefix before joining it to customer or seller records.


#### Column-Level Relationship Map

Each card is one source CSV and lists its original columns. Bold blue columns are join fields. Colored links group the join types; the exact field pairs and relationship cardinalities are listed in the table above. The raw geolocation file is not unique by postal-code prefix, so aggregate it to one row per prefix before using either location link.

In [1]:
# Visualize source columns and verified join fields using CSV headers only.
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go

schema_dir = Path("source_data")
schema_files = {
    "Geolocation": "olist_geolocation_dataset.csv",
    "Customers": "olist_customers_dataset.csv",
    "Orders": "olist_orders_dataset.csv",
    "Payments": "olist_order_payments_dataset.csv",
    "Reviews": "olist_order_reviews_dataset.csv",
    "Items": "olist_order_items_dataset.csv",
    "Translation": "product_category_name_translation.csv",
    "Products": "olist_products_dataset.csv",
    "Sellers": "olist_sellers_dataset.csv",
}
schema_columns = {
    name: pd.read_csv(schema_dir / filename, nrows=0).columns.tolist()
    for name, filename in schema_files.items()
}
schema_links = [
    ("Orders", "customer_id", "Customers", "customer_id", "Entity key"),
    ("Orders", "order_id", "Payments", "order_id", "Order key"),
    ("Orders", "order_id", "Reviews", "order_id", "Order key"),
    ("Orders", "order_id", "Items", "order_id", "Order key"),
    ("Items", "product_id", "Products", "product_id", "Entity key"),
    ("Items", "seller_id", "Sellers", "seller_id", "Entity key"),
    ("Products", "product_category_name", "Translation", "product_category_name", "Category key"),
    ("Customers", "customer_zip_code_prefix", "Geolocation", "geolocation_zip_code_prefix", "Postal prefix"),
    ("Sellers", "seller_zip_code_prefix", "Geolocation", "geolocation_zip_code_prefix", "Postal prefix"),
]
for left, left_key, right, right_key, _ in schema_links:
    assert left_key in schema_columns[left] and right_key in schema_columns[right]

schema_positions = {
    "Geolocation": (2, 65), "Customers": (26, 95), "Orders": (26, 68),
    "Payments": (51, 95), "Reviews": (51, 69), "Items": (51, 38),
    "Translation": (76, 95), "Products": (76, 77), "Sellers": (76, 38),
}
schema_colors = {
    "Order key": "#2563EB", "Entity key": "#7C3AED",
    "Category key": "#059669", "Postal prefix": "#EA580C",
}
join_columns = {
    name: {
        field for left, left_key, right, right_key, _ in schema_links
        for table, field in [(left, left_key), (right, right_key)]
        if table == name
    }
    for name in schema_files
}
box_width = 22
box_bottom = {
    name: top - (10 + 2.55 * len(schema_columns[name]))
    for name, (_, top) in schema_positions.items()
}
schema_figure = go.Figure()
shown_link_types = set()

def field_y(table, field):
    return schema_positions[table][1] - 10.5 - 2.55 * schema_columns[table].index(field)

for left, left_key, right, right_key, link_type in schema_links:
    lx, _ = schema_positions[left]
    rx, _ = schema_positions[right]
    ly, ry = field_y(left, left_key), field_y(right, right_key)
    if {left, right} == {"Sellers", "Geolocation"}:
        path_x, path_y = [lx, 74, 74, 1, 1, rx], [ly, ly, 4, 4, ry, ry]
    elif lx == rx:
        route_x = 24 if left == "Orders" else 99
        edge_x = lx if route_x < lx else lx + box_width
        path_x, path_y = [edge_x, route_x, route_x, edge_x], [ly, ly, ry, ry]
    elif lx < rx:
        path_x, path_y = [lx + box_width, rx], [ly, ry]
    else:
        path_x, path_y = [lx, rx + box_width], [ly, ry]
    schema_figure.add_trace(go.Scatter(
        x=path_x, y=path_y, mode="lines",
        line={"color": schema_colors[link_type], "width": 3},
        text=f"{left}.{left_key} ↔ {right}.{right_key}",
        hovertemplate="%{text}<extra></extra>",
        name=link_type, showlegend=link_type not in shown_link_types,
    ))
    shown_link_types.add(link_type)

for name, filename in schema_files.items():
    x, top = schema_positions[name]
    bottom = box_bottom[name]
    schema_figure.add_shape(
        type="rect", x0=x, x1=x + box_width, y0=bottom, y1=top,
        fillcolor="#FFFFFF", line={"color": "#94A3B8", "width": 1.5}, layer="above",
    )
    schema_figure.add_shape(
        type="rect", x0=x, x1=x + box_width, y0=top - 8, y1=top,
        fillcolor="#DBEAFE" if name == "Orders" else "#E2E8F0",
        line={"width": 0}, layer="above",
    )
    schema_figure.add_annotation(
        x=x + 1, y=top - 0.6, xanchor="left", yanchor="top",
        text=f"<b>{name}</b><br><span style='font-size:10px'>{filename}</span>",
        showarrow=False, align="left", font={"size": 14, "color": "#0F172A"},
    )
    field_lines = [
        f"<b><span style='color:#1D4ED8'>{field}</span></b>" if field in join_columns[name] else field
        for field in schema_columns[name]
    ]
    schema_figure.add_annotation(
        x=x + 1, y=top - 9.5, xanchor="left", yanchor="top",
        text="<br>".join(field_lines), showarrow=False, align="left",
        font={"size": 12, "color": "#334155"},
    )

schema_figure.update_xaxes(range=[0, 100], visible=False, fixedrange=True)
schema_figure.update_yaxes(range=[0, 100], visible=False, fixedrange=True)
schema_figure.update_layout(
    title="Olist CSV Column Relationships",
    template="plotly_white", width=1650, height=1250,
    legend={"orientation": "h", "x": 0.5, "xanchor": "center", "y": 1.01},
    margin={"l": 25, "r": 25, "t": 110, "b": 25},
    hovermode="closest",
)
schema_figure.show()

### 3.3 Data Dictionary

#### `olist_customers_dataset.csv`

| Field | Data Type | Business Meaning |
|---|---|---|
| `customer_id` | Text | Identifier for the customer record associated with an order. |
| `customer_unique_id` | Text | Persistent identifier for an individual customer across multiple orders. |
| `customer_zip_code_prefix` | Text | Customer postal-code prefix. |
| `customer_city` | Text | Customer city. |
| `customer_state` | Text | Customer state in Brazil. |

**Dataset of customers' information such as id and geolocations.**

#### `olist_geolocation_dataset.csv`

| Field | Data Type | Business Meaning |
|---|---|---|
| `geolocation_zip_code_prefix` | Text | Postal-code prefix used to locate the geographic record. |
| `geolocation_lat` | Decimal | Latitude of the geographic record. |
| `geolocation_lng` | Decimal | Longitude of the geographic record. |
| `geolocation_city` | Text | City associated with the geographic record. |
| `geolocation_state` | Text | Brazilian state associated with the geographic record. |

**Dataset of geolocation positioning.**

#### `olist_order_items_dataset.csv`

| Field | Data Type | Business Meaning |
|---|---|---|
| `order_id` | Text | Identifier of the order containing the item. |
| `order_item_id` | Integer | Sequential number of the item within an order. |
| `product_id` | Text | Identifier of the purchased product. |
| `seller_id` | Text | Identifier of the seller fulfilling the item. |
| `shipping_limit_date` | DateTime | Deadline for the seller to hand the item to the logistics partner. |
| `price` | Decimal | Item price charged to the customer. |
| `freight_value` | Decimal | Freight charge assigned to the order item. |

**Dataset of basic items' selling informations.**

#### `olist_order_payments_dataset.csv`

| Field | Data Type | Business Meaning |
|---|---|---|
| `order_id` | Text | Identifier of the order associated with the payment. |
| `payment_sequential` | Integer | Sequence number for a payment within an order. |
| `payment_type` | Text | Payment method used by the customer. |
| `payment_installments` | Integer | Number of installments selected for the payment. |
| `payment_value` | Decimal | Total monetary value of the payment record. |

**Dataset of orders' paying meathods and amounts.**

#### `olist_order_reviews_dataset.csv`

| Field | Data Type | Business Meaning |
|---|---|---|
| `review_id` | Text | Unique identifier for a customer review. |
| `order_id` | Text | Identifier of the order reviewed by the customer. |
| `review_score` | Integer | Customer satisfaction rating, typically on a five-point scale. |
| `review_comment_title` | Text | Title supplied by the customer for the review. |
| `review_comment_message` | Text | Written comment supplied by the customer. |
| `review_creation_date` | DateTime | Date and time when the customer created the review. |
| `review_answer_timestamp` | DateTime | Date and time when the marketplace responded to the review. |

**Dataset of orders' reviews' rattings, comments and timestamps.**

#### `olist_orders_dataset.csv`

| Field | Data Type | Business Meaning |
|---|---|---|
| `order_id` | Text | Unique identifier for an order. |
| `customer_id` | Text | Identifier of the customer record linked to the order. |
| `order_status` | Text | Current operational status of the order. |
| `order_purchase_timestamp` | DateTime | Date and time when the customer placed the order. |
| `order_approved_at` | DateTime | Date and time when the payment or order approval was completed. |
| `order_delivered_carrier_date` | DateTime | Date and time when the carrier received the order. |
| `order_delivered_customer_date` | DateTime | Date and time when the customer received the order. |
| `order_estimated_delivery_date` | DateTime | Estimated delivery date communicated for the order. |

**Dataset of orders' lifecycle infos.**

#### `olist_products_dataset.csv`

| Field | Data Type | Business Meaning |
|---|---|---|
| `product_id` | Text | Unique identifier for a product. |
| `product_category_name` | Text | Product category label in Portuguese. |
| `product_name_lenght` | Integer | Number of characters in the product name. |
| `product_description_lenght` | Integer | Number of characters in the product description. |
| `product_photos_qty` | Integer | Number of product photographs available. |
| `product_weight_g` | Decimal | Product weight in grams. |
| `product_length_cm` | Decimal | Product length in centimeters. |
| `product_height_cm` | Decimal | Product height in centimeters. |
| `product_width_cm` | Decimal | Product width in centimeters. |

**Dataset of online products infos.**

#### `olist_sellers_dataset.csv`

| Field | Data Type | Business Meaning |
|---|---|---|
| `seller_id` | Text | Unique identifier for a marketplace seller. |
| `seller_zip_code_prefix` | Text | Seller postal-code prefix. |
| `seller_city` | Text | Seller city. |
| `seller_state` | Text | Seller state in Brazil. |

**Dataset of sellers' infos.**

#### `product_category_name_translation.csv`

| Field | Data Type | Business Meaning |
|---|---|---|
| `product_category_name` | Text | Original Portuguese product category label. |
| `product_category_name_english` | Text | English translation of the product category label. |

**Dataset of product categories' English translations.**


## 4. Data Preparation


### 4.1 Environment Setup


In [2]:
import duckdb
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display
from pathlib import Path

DATA_DIR = Path("source_data")

### 4.2 Data Loading


In [3]:
customers_file = 'olist_customers_dataset.csv'
geolocation_file = 'olist_geolocation_dataset.csv'
items_file = 'olist_order_items_dataset.csv'
payments_file = 'olist_order_payments_dataset.csv'
reviews_file = 'olist_order_reviews_dataset.csv'
orders_file = 'olist_orders_dataset.csv'
products_file = 'olist_products_dataset.csv'
sellers_file = 'olist_sellers_dataset.csv'
translation_file = 'product_category_name_translation.csv'

files = [
    customers_file,
    geolocation_file,
    items_file,
    payments_file,
    reviews_file,
    orders_file,
    products_file,
    sellers_file,
    translation_file]

### 4.3 Data Quality Assessment


In [4]:
quality_results = []

for _ in files:
    df = pd.read_csv(DATA_DIR / _)
    
    total_row = df.shape[0]
    total_cells = df.shape[0]*df.shape[1]
    
    nulls = df.isnull().sum().sum()
    duplicates = df.duplicated().sum()
    
    missing_rate = nulls / total_cells if total_cells > 0 else 0
    duplicate_rate = duplicates / total_row if total_row > 0 else 0

    quality_results.append({
        "Dataset":_,
        "Total Rows": total_row,
        "Columns": df.shape[1],
        "Missing Values": nulls,
        "Missing Rate": missing_rate,
        "Duplicate Rows": duplicates,
        "Duplicate Rate":duplicate_rate
    })
    
quality_summary = pd.DataFrame(quality_results)

quality_summary['Missing Rate'] = quality_summary["Missing Rate"].map(lambda value:f"{value:.2%}")
quality_summary['Duplicate Rate'] = quality_summary['Duplicate Rate'].map(lambda value:f"{value:.2%}")

display(quality_summary)


,Dataset,Total Rows,Columns,Missing Values,Missing Rate,Duplicate Rows,Duplicate Rate
0,olist_customers_dataset.csv,99441,5,0,0.00%,0,0.00%
1,olist_geolocation_dataset.csv,1000163,5,0,0.00%,261831,26.18%
2,olist_order_items_dataset.csv,112650,7,0,0.00%,0,0.00%
3,olist_order_payments_dataset.csv,103886,5,0,0.00%,0,0.00%
4,olist_order_reviews_dataset.csv,99224,7,145903,21.01%,0,0.00%
5,olist_orders_dataset.csv,99441,8,4908,0.62%,0,0.00%
6,olist_products_dataset.csv,32951,9,2448,0.83%,0,0.00%
7,olist_sellers_dataset.csv,3095,4,0,0.00%,0,0.00%
8,product_category_name_translation.csv,71,2,0,0.00%,0,0.00%


For missing elements in reviews sheet are reasonable since not everyone leaves a review after buying a product, but missing elements in orders and products sheets are not as for every elements in these sheets are essential and indispensable.
For duplicate elements in geolocation sheet are also quite not reasonable since the exact same rows have no additional meanings so those could be cleaned.

### 4.4 Data Cleaning


In [5]:
file_map = {
    "customers_df": "olist_customers_dataset.csv",
    "geolocation_df": "olist_geolocation_dataset.csv",
    "items_df": "olist_order_items_dataset.csv",
    "payments_df": "olist_order_payments_dataset.csv",
    "reviews_df": "olist_order_reviews_dataset.csv",
    "orders_df": "olist_orders_dataset.csv",
    "products_df": "olist_products_dataset.csv",
    "sellers_df": "olist_sellers_dataset.csv",
    "translation_df": "product_category_name_translation.csv",
}

dfs: dict[str, pd.DataFrame] = {}

for df_name, file_name in file_map.items():
    dfs[df_name] = pd.read_csv(DATA_DIR / file_name)

In [6]:
quality_results = []

for df_name, df in dfs.items():
    total_rows = df.shape[0]
    total_columns = df.shape[1]
    total_cells = total_rows * total_columns

    missing_values = df.isnull().sum().sum()
    duplicate_rows = df.duplicated().sum()

    missing_rate = (
        missing_values / total_cells
        if total_cells > 0
        else 0
    )

    duplicate_rate = (
        duplicate_rows / total_rows
        if total_rows > 0
        else 0
    )

    quality_results.append({
        "Dataframe": df_name,
        "Rows": total_rows,
        "Columns": total_columns,
        "Missing Values": missing_values,
        "Missing Rate": missing_rate,
        "Duplicate Rows": duplicate_rows,
        "Duplicate Rate": duplicate_rate,
    })

quality_summary = pd.DataFrame(quality_results)

quality_summary["Missing Rate"] = quality_summary["Missing Rate"].map(
    lambda value: f"{value:.2%}"
)

quality_summary["Duplicate Rate"] = quality_summary["Duplicate Rate"].map(
    lambda value: f"{value:.2%}"
)

display(quality_summary)

,Dataframe,Rows,Columns,Missing Values,Missing Rate,Duplicate Rows,Duplicate Rate
0,customers_df,99441,5,0,0.00%,0,0.00%
1,geolocation_df,1000163,5,0,0.00%,261831,26.18%
2,items_df,112650,7,0,0.00%,0,0.00%
3,payments_df,103886,5,0,0.00%,0,0.00%
4,reviews_df,99224,7,145903,21.01%,0,0.00%
5,orders_df,99441,8,4908,0.62%,0,0.00%
6,products_df,32951,9,2448,0.83%,0,0.00%
7,sellers_df,3095,4,0,0.00%,0,0.00%
8,translation_df,71,2,0,0.00%,0,0.00%


In [7]:
customers_df = dfs["customers_df"]
geolocation_df = dfs["geolocation_df"]
items_df = dfs["items_df"]
payments_df = dfs["payments_df"]
reviews_df = dfs["reviews_df"]
orders_df = dfs["orders_df"]
products_df = dfs["products_df"]
sellers_df = dfs["sellers_df"]
translation_df = dfs["translation_df"]

### 4.5 Feature Engineering


In [8]:
# Customers Features
customers_df["customer_zip_code_prefix"] = customers_df["customer_zip_code_prefix"].astype("string").str.zfill(5)
customers_df["customer_city"] = customers_df["customer_city"].str.strip().str.lower()

"str-lize zip code prefixes and city names."

# Geolocation Features
geolocation_df["geolocation_zip_code_prefix"] = geolocation_df["geolocation_zip_code_prefix"].astype("string").str.zfill(5)
geolocation_df["geolocation_lat"] = pd.to_numeric(geolocation_df["geolocation_lat"], errors="coerce")
geolocation_df["geolocation_lng"] = pd.to_numeric(geolocation_df["geolocation_lng"], errors="coerce")

"numerilize geolocation's lat and lng values."

# Order Item Features
items_df["shipping_limit_date"] = pd.to_datetime(items_df["shipping_limit_date"], errors="coerce")
items_df["price"] = pd.to_numeric(items_df["price"], errors="coerce")
items_df["freight_value"] = pd.to_numeric(items_df["freight_value"], errors="coerce")
items_df["item_total_value"] = items_df["price"] + items_df["freight_value"]

"normalize timestamps and numerilize values."

# Payment Features
payments_df["payment_value"] = pd.to_numeric(payments_df["payment_value"], errors="coerce")
payments_df["payment_installments"] = pd.to_numeric(payments_df["payment_installments"], errors="coerce")

"numerilize values"

# Review Features
reviews_df["review_creation_date"] = pd.to_datetime(reviews_df["review_creation_date"], errors="coerce")
reviews_df["review_answer_timestamp"] = pd.to_datetime(reviews_df["review_answer_timestamp"], errors="coerce")
reviews_df["review_score"] = pd.to_numeric(reviews_df["review_score"], errors="coerce")

"normalize timestamps and numerilize review scores."

# Order Features
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for column in date_columns:
    orders_df[column] = pd.to_datetime(orders_df[column], errors="coerce")
orders_df["purchase_year"] = orders_df["order_purchase_timestamp"].dt.year
orders_df["purchase_month"] = orders_df["order_purchase_timestamp"].dt.to_period("M").astype("string")
orders_df["delivery_days"] = (orders_df["order_delivered_customer_date"] - orders_df["order_purchase_timestamp"]).dt.total_seconds() / 86400
orders_df["delivery_delay_days"] = (orders_df["order_delivered_customer_date"] - orders_df["order_estimated_delivery_date"]).dt.total_seconds() / 86400
orders_df["is_late_delivery"] = orders_df["delivery_delay_days"] > 0

"slice timestamps by year, month and day, creating new features."

# Product Features
products_df["product_category_name"] = products_df["product_category_name"].str.strip().str.lower()
product_numeric_columns = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
]
for column in product_numeric_columns:
    products_df[column] = pd.to_numeric(products_df[column], errors="coerce")

"numerilize products' parameters."

# Seller Features
sellers_df["seller_zip_code_prefix"] = sellers_df["seller_zip_code_prefix"].astype("string").str.zfill(5)
sellers_df["seller_city"] = sellers_df["seller_city"].str.strip().str.lower()

"str-lize zip codes and city names."

""


''

## 5. Macro Olist Market Analysis


### 5.1 Market GMV, Average Order Value and Order Volume


In [9]:
# Calculate market-scale KPIs from delivered orders
delivered_order_ids = orders_df.loc[
    orders_df["order_status"].eq("delivered"), "order_id"
].unique()

delivered_items = items_df[items_df["order_id"].isin(delivered_order_ids)]
order_volume = len(delivered_order_ids)
gmv = delivered_items["price"].sum()
average_order_value = gmv / order_volume if order_volume else 0
delivered_customers = orders_df.loc[
    orders_df["order_status"].eq("delivered"), ["customer_id"]
].merge(customers_df[["customer_id", "customer_unique_id"]], on="customer_id", how="left")
transacting_customers = delivered_customers["customer_unique_id"].nunique()
active_sellers = delivered_items["seller_id"].nunique()

kpi_figure = go.Figure(
    data=[
        go.Indicator(
            mode="number",
            value=gmv,
            number={"prefix": "R$ ", "valueformat": ",.0f", "font": {"size": 36, "color": "#2563EB"}},
            title={"text": "GMV", "font": {"size": 16}},
            domain={"x": [0.0, 0.30], "y": [0.55, 1]},
        ),
        go.Indicator(
            mode="number",
            value=order_volume,
            number={"valueformat": ",.0f", "font": {"size": 36, "color": "#059669"}},
            title={"text": "Delivered Orders", "font": {"size": 16}},
            domain={"x": [0.35, 0.65], "y": [0.55, 1]},
        ),
        go.Indicator(
            mode="number",
            value=average_order_value,
            number={"prefix": "R$ ", "valueformat": ",.2f", "font": {"size": 36, "color": "#EA580C"}},
            title={"text": "Average Order Value", "font": {"size": 16}},
            domain={"x": [0.70, 1.0], "y": [0.55, 1]},
        ),
        go.Indicator(
            mode="number",
            value=transacting_customers,
            number={"valueformat": ",.0f", "font": {"size": 36, "color": "#7C3AED"}},
            title={"text": "Transacting Customers", "font": {"size": 16}},
            domain={"x": [0.18, 0.45], "y": [0, 0.35]},
        ),
        go.Indicator(
            mode="number",
            value=active_sellers,
            number={"valueformat": ",.0f", "font": {"size": 36, "color": "#DB2777"}},
            title={"text": "Active Sellers", "font": {"size": 16}},
            domain={"x": [0.55, 0.82], "y": [0, 0.35]},
        ),
    ]
)

kpi_figure.update_layout(
    title="Olist Marketplace GMV and Order Scale",
    height=420,
    margin={"l": 30, "r": 30, "t": 70, "b": 30},
)
kpi_figure.show()

**From September 2016 to August 2018, Olist's GMV in Brazil was R$13,221,498, approximately US$2,650,000. The platform delivered 96,478 orders, served 93,358 transacting customers, and had 2,970 active sellers. The average order value was R$137, approximately US$27.50.**

### 5.2 Growth Trends and Seasonality


In [10]:
# Build monthly GMV and delivered-order time series
delivered_orders_ts = orders_df.loc[
    orders_df["order_status"].eq("delivered"),
    ["order_id", "order_purchase_timestamp"],
].copy()
delivered_orders_ts["month"] = delivered_orders_ts["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp()

monthly_orders = (
    delivered_orders_ts.groupby("month", as_index=False)
    .agg(delivered_orders=("order_id", "nunique"))
)
monthly_gmv = (
    items_df.merge(delivered_orders_ts[["order_id", "month"]], on="order_id", how="inner")
    .groupby("month", as_index=False)
    .agg(gmv=("price", "sum"))
)
monthly_market = (
    monthly_orders.merge(monthly_gmv, on="month", how="outer")
    .fillna(0)
    .sort_values("month")
)

# Calculate linear regression trend lines using actual elapsed days
elapsed_days = (monthly_market["month"] - monthly_market["month"].min()).dt.days.to_numpy()
gmv_slope, gmv_intercept = np.polyfit(elapsed_days, monthly_market["gmv"], 1)
orders_slope, orders_intercept = np.polyfit(elapsed_days, monthly_market["delivered_orders"], 1)
monthly_market["gmv_trend"] = gmv_slope * elapsed_days + gmv_intercept
monthly_market["orders_trend"] = orders_slope * elapsed_days + orders_intercept
gmv_r_squared = 1 - ((monthly_market["gmv"] - monthly_market["gmv_trend"]) ** 2).sum() / ((monthly_market["gmv"] - monthly_market["gmv"].mean()) ** 2).sum()
orders_r_squared = 1 - ((monthly_market["delivered_orders"] - monthly_market["orders_trend"]) ** 2).sum() / ((monthly_market["delivered_orders"] - monthly_market["delivered_orders"].mean()) ** 2).sum()
label_index = max(len(monthly_market) - 4, 0)
orders_axis_max = monthly_market["delivered_orders"].max() * 1.4

# Plot monthly GMV and delivered orders with dual Y-axes and regression lines
growth_figure = go.Figure()
growth_figure.add_trace(
    go.Scatter(
        x=monthly_market["month"],
        y=monthly_market["gmv"],
        mode="lines",
        name="GMV",
        line={"color": "#2563EB", "width": 3},
    )
)
growth_figure.add_trace(
    go.Scatter(
        x=monthly_market["month"],
        y=monthly_market["gmv_trend"],
        mode="lines",
        name="GMV Regression Trend",
        line={"color": "#93C5FD", "width": 2, "dash": "dash"},
    )
)
growth_figure.add_trace(
    go.Scatter(
        x=monthly_market["month"],
        y=monthly_market["delivered_orders"],
        mode="lines",
        name="Delivered Orders",
        line={"color": "#059669", "width": 3},
        yaxis="y2",
    )
)
growth_figure.add_trace(
    go.Scatter(
        x=monthly_market["month"],
        y=monthly_market["orders_trend"],
        mode="lines",
        name="Delivered Orders Regression Trend",
        line={"color": "#6EE7B7", "width": 2, "dash": "dash"},
        yaxis="y2",
    )
)
growth_figure.update_layout(
    title="Monthly GMV and Delivered Orders Growth Trend",
    xaxis={"title": "Month"},
    yaxis={"title": {"text": "GMV (R$)", "font": {"color": "#2563EB"}}, "tickfont": {"color": "#2563EB"}},
    yaxis2={"title": {"text": "Delivered Orders", "font": {"color": "#059669"}}, "tickfont": {"color": "#059669"}, "range": [0, orders_axis_max], "overlaying": "y", "side": "right"},
    template="plotly_white",
    height=550,
    legend={"orientation": "h", "y": 1.12},
    annotations=[
        {"x": monthly_market["month"].iloc[label_index], "y": monthly_market["gmv_trend"].iloc[label_index], "text": f"GMV Trend<br>R² = {gmv_r_squared:.2f}", "showarrow": False, "font": {"color": "#2563EB"}, "yshift": 18},
        {"x": monthly_market["month"].iloc[label_index], "y": monthly_market["orders_trend"].iloc[label_index], "yref": "y2", "text": f"Orders Trend<br>R² = {orders_r_squared:.2f}", "showarrow": False, "font": {"color": "#059669"}, "yshift": -20},
    ],
)
growth_figure.show()

**The chart shows that Olist's GMV and delivered-order volume generally increased from September 2016 to August 2018. In the latest month, the platform recorded approximately R$839,000 in GMV and 6,351 delivered orders. The linear regression trends rise by approximately R$45,700 in GMV and 334 orders per month. The R-squared values for both series indicate a relatively good linear fit. Together, these results suggest broad prospects for Olist's expansion in Brazil.**

### 5.2.1 Three-Month Multivariable Regression Forecast

This exploratory forecast uses complete months from January 2017 through August 2018. For each target, the linear model uses a year-and-month time index plus month-of-year seasonality:

$$
y_m=\beta_0+\beta_1[12(\mathrm{Year}_m-2017)+(\mathrm{Month}_m-1)]
+\beta_2\sin\!\left(\frac{2\pi(\mathrm{Month}_m-1)}{12}\right)
+\beta_3\cos\!\left(\frac{2\pi(\mathrm{Month}_m-1)}{12}\right)+\varepsilon_m.
$$

The shaded 95% confidence interval describes uncertainty in the **expected monthly mean**, not a range guaranteed to contain an individual future month. With only 20 historical months, trend changes, autocorrelation, and external shocks are not captured.

In [11]:
# Forecast three months from the last complete observed month.
from plotly.subplots import make_subplots
from scipy.stats import t as student_t

forecast_history = monthly_market.loc[
    monthly_market["month"].between(pd.Timestamp("2017-01-01"), pd.Timestamp("2018-08-01"))
].copy()
expected_months = pd.date_range("2017-01-01", "2018-08-01", freq="MS")
assert forecast_history["month"].tolist() == expected_months.tolist(), "Training months must be complete and consecutive."
forecast_months = pd.date_range(
    forecast_history["month"].max() + pd.offsets.MonthBegin(1), periods=3, freq="MS"
)

def monthly_design(months):
    year = months.dt.year.to_numpy()
    month = months.dt.month.to_numpy()
    elapsed_months = (year - 2017) * 12 + (month - 1)
    return np.column_stack([
        np.ones(len(months)), elapsed_months,
        np.sin(2 * np.pi * (month - 1) / 12),
        np.cos(2 * np.pi * (month - 1) / 12),
    ])

history_design = monthly_design(forecast_history["month"])
future_design = monthly_design(pd.Series(forecast_months))
forecast_table = pd.DataFrame({"month": forecast_months})

for target in ["gmv", "delivered_orders"]:
    observed = forecast_history[target].to_numpy(dtype=float)
    coefficients = np.linalg.lstsq(history_design, observed, rcond=None)[0]
    fitted = history_design @ coefficients
    residual_degrees = len(observed) - history_design.shape[1]
    residual_variance = np.sum((observed - fitted) ** 2) / residual_degrees
    inverse_information = np.linalg.pinv(history_design.T @ history_design)
    expected_values = future_design @ coefficients
    mean_standard_error = np.sqrt(
        residual_variance * np.einsum("ij,jk,ik->i", future_design, inverse_information, future_design)
    )
    confidence_margin = student_t.ppf(0.975, residual_degrees) * mean_standard_error
    forecast_table[target] = expected_values
    forecast_table[f"{target}_ci_low"] = expected_values - confidence_margin
    forecast_table[f"{target}_ci_high"] = expected_values + confidence_margin

forecast_figure = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12,
    subplot_titles=("GMV", "Delivered Orders"),
)
for row, target, color, fill_color in [
    (1, "gmv", "#2563EB", "rgba(37,99,235,0.17)"),
    (2, "delivered_orders", "#059669", "rgba(5,150,105,0.17)"),
]:
    forecast_figure.add_trace(go.Scatter(
        x=forecast_history["month"], y=forecast_history[target],
        mode="lines+markers", name=f"Observed {target.replace('_', ' ').title()}",
        line={"color": color, "width": 2.5},
    ), row=row, col=1)
    forecast_figure.add_trace(go.Scatter(
        x=forecast_months, y=forecast_table[f"{target}_ci_low"],
        mode="lines", line={"color": color, "width": 0}, showlegend=False, hoverinfo="skip",
    ), row=row, col=1)
    forecast_figure.add_trace(go.Scatter(
        x=forecast_months, y=forecast_table[f"{target}_ci_high"],
        mode="lines", line={"color": color, "width": 0}, fill="tonexty",
        fillcolor=fill_color, name="95% Mean CI" if row == 1 else None,
        showlegend=row == 1, hoverinfo="skip",
    ), row=row, col=1)
    forecast_figure.add_trace(go.Scatter(
        x=[forecast_history["month"].iloc[-1], *forecast_months],
        y=[forecast_history[target].iloc[-1], *forecast_table[target].tolist()],
        mode="lines+markers", name=f"Forecast {target.replace('_', ' ').title()}",
        line={"color": color, "width": 3, "dash": "dash"},
        marker={"symbol": "diamond", "size": 8},
    ), row=row, col=1)
forecast_figure.update_yaxes(title_text="GMV (R$)", row=1, col=1)
forecast_figure.update_yaxes(title_text="Orders", row=2, col=1)
forecast_figure.update_xaxes(title_text="Month", row=2, col=1)
forecast_figure.update_layout(
    title="Three-Month Historical Forecast from August 2018",
    template="plotly_white", height=760,
    legend={"orientation": "h", "y": 1.08}, margin={"l": 75, "r": 35, "t": 110, "b": 65},
)
forecast_figure.show()
display(forecast_table.style.format({
    "gmv": "R$ {:,.0f}", "gmv_ci_low": "R$ {:,.0f}", "gmv_ci_high": "R$ {:,.0f}",
    "delivered_orders": "{:,.0f}", "delivered_orders_ci_low": "{:,.0f}",
    "delivered_orders_ci_high": "{:,.0f}",
}))

,month,gmv,gmv_ci_low,gmv_ci_high,delivered_orders,delivered_orders_ci_low,delivered_orders_ci_high
0,2018-09-01 00:00:00,"R$ 1,099,896","R$ 960,680","R$ 1,239,113","8,088","7,038","9,139"
1,2018-10-01 00:00:00,"R$ 1,177,456","R$ 1,019,997","R$ 1,334,916","8,708","7,520","9,896"
2,2018-11-01 00:00:00,"R$ 1,252,709","R$ 1,078,870","R$ 1,426,548","9,293","7,982","10,605"


**Using a model with year and month variables, the forecast covers September through November 2018. It projects GMV of approximately R$1.10 million, R$1.18 million, and R$1.25 million, and about 8,088, 8,708, and 9,293 delivered orders, respectively. Across the three forecast months, GMV rises by roughly R$76,000 per month on average and orders by about 603 per month. The 95% confidence bands have total widths of approximately R$278,000–R$348,000 for GMV and 2,100–2,600 orders. The forecast suggests that Olist's Brazilian market performance will remain relatively stable while its estimated market scale continues to grow.**

### 5.3 Market Structure and Seller Competition


In [12]:
# Prepare delivered-order item data for GMV market-share analysis
delivered_order_ids = orders_df.loc[orders_df["order_status"].eq("delivered"), "order_id"]
market_items = items_df[items_df["order_id"].isin(delivered_order_ids)].copy()

# Plot product-category market share by GMV
category_share = (
    market_items.merge(products_df[["product_id", "product_category_name"]], on="product_id", how="left")
    .merge(translation_df, on="product_category_name", how="left")
    .assign(category_name=lambda dataframe: dataframe["product_category_name_english"].fillna(dataframe["product_category_name"]).fillna("Unknown"))
    .groupby("category_name", as_index=False)["price"]
    .sum()
    .rename(columns={"price": "gmv"})
    .sort_values("gmv", ascending=False)
)
category_share["market_share"] = category_share["gmv"] / category_share["gmv"].sum()

category_treemap = px.treemap(
    category_share,
    path=["category_name"],
    values="gmv",
    color="gmv",
    color_continuous_scale="Blues",
    custom_data=["market_share"],
    title="Product Category GMV Market Share",
)
category_treemap.update_traces(
    texttemplate="%{label}<br>%{customdata[0]:.1%}",
    hovertemplate="<b>%{label}</b><br>GMV: R$ %{value:,.0f}<br>Market Share: %{customdata[0]:.2%}<extra></extra>",
)
category_treemap.update_layout(height=650, margin={"t": 60, "l": 20, "r": 20, "b": 20})
category_treemap.show()


**The treemap shows that the largest product categories in Olist's Brazilian sales are health and beauty, watches and gifts, bed and bath, sports and leisure, computer accessories, and furniture and decor. Each contributes more than 5% of GMV. The marketplace offers a wide range of product categories, and Olist is becoming more diversified.**

In [13]:
# Build a product-category Pareto table from delivered-order GMV
category_pareto = category_share.sort_values("gmv", ascending=False).reset_index(drop=True).copy()
category_pareto["cumulative_share"] = category_pareto["gmv"].cumsum() / category_pareto["gmv"].sum()
pareto_80_category_count = int(category_pareto["cumulative_share"].ge(0.80).idxmax() + 1)
pareto_80_category_percentage = pareto_80_category_count / len(category_pareto)

# Plot category GMV and cumulative market share with an 80% reference line
category_pareto_figure = go.Figure()
category_pareto_figure.add_trace(
    go.Bar(
        x=category_pareto["category_name"],
        y=category_pareto["gmv"],
        name="Category GMV",
        marker={"color": "#2563EB"},
        customdata=category_pareto[["market_share"]],
        hovertemplate="<b>%{x}</b><br>GMV: R$ %{y:,.0f}<br>Market Share: %{customdata[0]:.2%}<extra></extra>",
    )
)
category_pareto_figure.add_trace(
    go.Scatter(
        x=category_pareto["category_name"],
        y=category_pareto["cumulative_share"],
        name="Cumulative GMV Share",
        mode="lines",
        line={"color": "#DC2626", "width": 3},
        marker={"size": 6},
        yaxis="y2",
        hovertemplate="<b>%{x}</b><br>Cumulative Share: %{y:.2%}<extra></extra>",
    )
)
category_pareto_figure.update_layout(
    title=f"Product Category GMV Pareto Analysis: {pareto_80_category_percentage:.1%} of Categories Generate 80% of GMV",
    xaxis={"title": "Product Category", "tickangle": -60, "automargin": True},
    yaxis={"title": {"text": "GMV (R$)", "font": {"color": "#2563EB"}}, "tickfont": {"color": "#2563EB"}},
    yaxis2={"title": {"text": "Cumulative GMV Share", "font": {"color": "#DC2626"}}, "tickfont": {"color": "#DC2626"}, "tickformat": ".0%", "range": [0, 1.05], "overlaying": "y", "side": "right"},
    template="plotly_white",
    height=700,
    legend={"orientation": "h", "y": 1.10},
    shapes=[
        {"type": "line", "xref": "paper", "x0": 0, "x1": 1, "yref": "y2", "y0": 0.80, "y1": 0.80, "line": {"color": "#F59E0B", "width": 2, "dash": "dash"}}
    ],
    annotations=[
        {"xref": "paper", "x": 0.99, "yref": "y2", "y": 0.80, "text": "80% Threshold", "showarrow": False, "xanchor": "right", "yanchor": "bottom", "font": {"color": "#B45309"}}
    ],
    margin={"t": 90, "l": 60, "r": 70, "b": 180},
)
category_pareto_figure.show()

**The top 18 of 74 product categories (24.3%) account for approximately 80% of delivered-item GMV in the Olist dataset.**

### 5.4 Payment Behavior and Consumer Spending


In [14]:
from plotly.subplots import make_subplots

# Exclude undefined payment records from the payment analysis
valid_payments = payments_df.loc[
    payments_df["payment_type"].fillna("").str.lower().ne("not_defined")
].copy()

# Summarize total payment value across all defined payment methods
payment_behavior = (
    valid_payments.groupby("payment_type", as_index=False)["payment_value"]
    .sum()
    .sort_values("payment_value", ascending=False)
)
payment_behavior["payment_method"] = (
    payment_behavior["payment_type"].str.replace("_", " ", regex=False).str.title()
)

# Calculate period-normalized monthly and annual spend per transacting consumer
consumer_payments = (
    valid_payments[["order_id", "payment_value"]]
    .merge(
        orders_df[["order_id", "customer_id", "order_purchase_timestamp"]],
        on="order_id",
        how="inner",
    )
    .merge(
        customers_df[["customer_id", "customer_unique_id"]],
        on="customer_id",
        how="inner",
    )
    .dropna(subset=["customer_unique_id", "order_purchase_timestamp", "payment_value"])
)
consumer_payments["purchase_month"] = consumer_payments["order_purchase_timestamp"].dt.to_period("M")
consumer_payments["purchase_year"] = consumer_payments["order_purchase_timestamp"].dt.year

# Use 2017 because it is the only complete calendar year in the dataset
consumer_payments_2017 = consumer_payments.loc[consumer_payments["purchase_year"].eq(2017)].copy()
monthly_consumer_spend = (
    consumer_payments_2017.groupby(["customer_unique_id", "purchase_month"])["payment_value"]
    .sum()
)
annual_consumer_spend = (
    consumer_payments_2017.groupby("customer_unique_id")["payment_value"]
    .sum()
)
average_monthly_consumer_spend = monthly_consumer_spend.mean()
average_annual_consumer_spend = annual_consumer_spend.mean()
consumer_order_counts = consumer_payments_2017.groupby("customer_unique_id")["order_id"].nunique()
repeat_purchase_rate = consumer_order_counts.gt(1).mean()

# Combine the payment-method chart with three vertically stacked consumer indicators
payment_figure = make_subplots(
    rows=3,
    cols=2,
    specs=[
        [{"type": "xy", "rowspan": 3}, {"type": "domain"}],
        [None, {"type": "domain"}],
        [None, {"type": "domain"}],
    ],
    column_widths=[0.68, 0.32],
    row_heights=[1 / 3, 1 / 3, 1 / 3],
    horizontal_spacing=0.12,
    vertical_spacing=0.08,
)

payment_colors = ["#2563EB", "#0F766E", "#F59E0B", "#7C3AED", "#64748B"]
payment_figure.add_trace(
    go.Bar(
        x=payment_behavior["payment_method"],
        y=payment_behavior["payment_value"],
        marker={"color": payment_colors[:len(payment_behavior)]},
        text=payment_behavior["payment_value"],
        texttemplate="R$ %{text:,.0f}",
        textposition="outside",
        cliponaxis=False,
        hovertemplate="<b>%{x}</b><br>Total Payment Value: R$ %{y:,.2f}<extra></extra>",
        showlegend=False,
    ),
    row=1,
    col=1,
)

payment_figure.add_trace(
    go.Indicator(
        mode="number",
        value=average_monthly_consumer_spend,
        number={"prefix": "R$ ", "valueformat": ",.2f", "font": {"size": 38, "color": "#2563EB"}},
        title={"text": "2017 Average Monthly Spend<br>per Active Consumer", "font": {"size": 16, "color": "#334155"}},
    ),
    row=1,
    col=2,
)
payment_figure.add_trace(
    go.Indicator(
        mode="number",
        value=average_annual_consumer_spend,
        number={"prefix": "R$ ", "valueformat": ",.2f", "font": {"size": 38, "color": "#EA580C"}},
        title={"text": "2017 Average Annual Spend<br>per Active Consumer", "font": {"size": 16, "color": "#334155"}},
    ),
    row=2,
    col=2,
)
payment_figure.add_trace(
    go.Indicator(
        mode="number",
        value=repeat_purchase_rate,
        number={"valueformat": ".1%", "font": {"size": 38, "color": "#7C3AED"}},
        title={"text": "2017 Repeat Purchase Rate<br>among Active Consumers", "font": {"size": 16, "color": "#334155"}},
    ),
    row=3,
    col=2,
)

payment_figure.update_layout(
    title="Payment Behavior and Consumer Spending",
    template="plotly_white",
    height=650,
    bargap=0.35,
    margin={"l": 70, "r": 35, "t": 85, "b": 70},
    font={"family": "Arial", "color": "#334155"},
    shapes=[
        {"type": "rect", "xref": "paper", "yref": "paper", "x0": 0.73, "x1": 1.00, "y0": 0.70, "y1": 1.00, "fillcolor": "#EFF6FF", "line": {"width": 0}, "layer": "below"},
        {"type": "rect", "xref": "paper", "yref": "paper", "x0": 0.73, "x1": 1.00, "y0": 0.35, "y1": 0.65, "fillcolor": "#FFF7ED", "line": {"width": 0}, "layer": "below"},
        {"type": "rect", "xref": "paper", "yref": "paper", "x0": 0.73, "x1": 1.00, "y0": 0.00, "y1": 0.30, "fillcolor": "#F5F3FF", "line": {"width": 0}, "layer": "below"},
    ],
)
payment_figure.update_xaxes(title_text="Payment Method", row=1, col=1)
payment_figure.update_yaxes(title_text="Total Payment Value (R$)", tickprefix="R$ ", tickformat=",.0f", rangemode="tozero", row=1, col=1)
payment_figure.show()


**The chart shows that credit cards are the leading payment method by total payment value, followed by boleto, vouchers, and debit cards. Boleto is a distinctive Brazilian offline payment method, indicating that online payment is widespread while offline payment options continue to fill a gap. Based on 2017 data, active customers spent an average of R$163.51 in an active month and R$165.85 in the year. This suggests that most customers purchased on the platform only once during that year and that the core repeat-customer group was small. The repeat-purchase rate among active customers was 2.87%.**

## 6. Product Analysis


### 6.1 Category Demand and Sales Mix


In [15]:
# Build an item-level delivered-order table with English category names
delivered_order_details = orders_df.loc[
    orders_df["order_status"].eq("delivered"),
    ["order_id", "customer_id", "order_purchase_timestamp", "delivery_days", "delivery_delay_days", "is_late_delivery"],
].copy()
product_sales = (
    items_df[items_df["order_id"].isin(delivered_order_details["order_id"])]
    .merge(products_df[["product_id", "product_category_name"]], on="product_id", how="left")
    .merge(translation_df, on="product_category_name", how="left")
    .merge(delivered_order_details[["order_id", "order_purchase_timestamp"]], on="order_id", how="left")
)
product_sales["category_name"] = (
    product_sales["product_category_name_english"]
    .fillna(product_sales["product_category_name"])
    .fillna("Unknown")
)
category_performance = (
    product_sales.groupby("category_name", as_index=False)
    .agg(
        gmv=("price", "sum"),
        orders=("order_id", "nunique"),
        units=("order_item_id", "count"),
        avg_item_price=("price", "mean"),
    )
)
category_performance["gmv_share"] = category_performance["gmv"] / category_performance["gmv"].sum()

top_gmv_categories = category_performance.nlargest(12, "gmv").sort_values("gmv")
top_order_categories = category_performance.nlargest(12, "orders").sort_values("orders")
category_demand_figure = make_subplots(rows=1, cols=2, subplot_titles=("Top Categories by GMV", "Top Categories by Delivered Orders"), horizontal_spacing=0.22)
category_demand_figure.add_trace(go.Bar(x=top_gmv_categories["gmv"], y=top_gmv_categories["category_name"], orientation="h", marker_color="#2563EB", text=top_gmv_categories["gmv"], texttemplate="R$ %{text:,.0f}", hovertemplate="<b>%{y}</b><br>GMV: R$ %{x:,.0f}<extra></extra>", showlegend=False), row=1, col=1)
category_demand_figure.add_trace(go.Bar(x=top_order_categories["orders"], y=top_order_categories["category_name"], orientation="h", marker_color="#0F766E", text=top_order_categories["orders"], texttemplate="%{text:,.0f}", hovertemplate="<b>%{y}</b><br>Orders: %{x:,.0f}<extra></extra>", showlegend=False), row=1, col=2)
category_demand_figure.update_layout(title="Category Demand and Sales Mix", template="plotly_white", height=620, margin={"l": 170, "r": 30, "t": 90, "b": 50})
category_demand_figure.update_xaxes(rangemode="tozero")
category_demand_figure.show()


### 6.2 Product Pricing and Freight Economics


In [16]:
# Compare category pricing with freight burden
category_economics = (
    product_sales.loc[product_sales["price"].gt(0)]
    .groupby("category_name", as_index=False)
    .agg(
        items=("order_item_id", "count"),
        median_price=("price", "median"),
        average_price=("price", "mean"),
        average_freight=("freight_value", "mean"),
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum"),
    )
)
category_economics["freight_to_price_ratio"] = category_economics["total_freight"] / category_economics["total_price"]
category_economics_view = category_economics.loc[category_economics["items"].ge(100)].copy()
pricing_figure = px.scatter(
    category_economics_view,
    x="median_price",
    y="freight_to_price_ratio",
    size="total_price",
    color="average_freight",
    hover_name="category_name",
    color_continuous_scale="Blues",
    title="Category Pricing and Freight Burden",
    labels={"median_price": "Median Item Price (R$)", "freight_to_price_ratio": "Freight-to-Price Ratio", "average_freight": "Average Freight (R$)", "total_price": "Category GMV (R$)"},
)

pricing_figure.update_yaxes(tickformat=".0%", rangemode="tozero")
pricing_figure.update_xaxes(rangemode="tozero")
pricing_figure.update_layout(template="plotly_white", height=560)
pricing_figure.show()


**The displayed categories have median item prices ranging from about R$22 to R$1,100 and freight-to-price ratios ranging from about 4.4% to 36.5%. Freight represents a substantial burden for some product categories, so improving delivery efficiency may be important when expanding in Brazil. Computers stands far to the right of the chart, with the highest median price and the lowest freight-to-price ratio. Despite its smaller GMV, it appears to be a favorable high-margin option. Considering the factors together, watches and gifts, cool stuff, and health and beauty also appear to offer attractive online sales opportunities and could be entry categories for the Brazilian market.**

### 6.3 Product Portfolio and Seller Competitiveness


In [17]:
# Measure product breadth, seller coverage, and seller concentration by category
seller_category_gmv = (
    product_sales.groupby(["category_name", "seller_id"], as_index=False)["price"]
    .sum()
    .rename(columns={"price": "seller_gmv"})
)
seller_category_gmv["seller_share"] = seller_category_gmv["seller_gmv"] / seller_category_gmv.groupby("category_name")["seller_gmv"].transform("sum")
seller_concentration = (
    seller_category_gmv.assign(squared_share=lambda dataframe: dataframe["seller_share"] ** 2)
    .groupby("category_name", as_index=False)["squared_share"]
    .sum()
    .rename(columns={"squared_share": "seller_hhi"})
)
category_portfolio = (
    product_sales.groupby("category_name", as_index=False)
    .agg(gmv=("price", "sum"), unique_products=("product_id", "nunique"), unique_sellers=("seller_id", "nunique"), orders=("order_id", "nunique"))
    .merge(seller_concentration, on="category_name", how="left")
)
portfolio_figure = px.scatter(
    category_portfolio.loc[category_portfolio["orders"].ge(50)],
    x="unique_sellers",
    y="unique_products",
    size="gmv",
    color="seller_hhi",
    hover_name="category_name",
    color_continuous_scale="Oranges",
    log_x=True,
    title="Product Portfolio Breadth and Seller Competition",
    labels={"unique_sellers": "Unique Sellers (log scale)", "unique_products": "Unique Products", "gmv": "Category GMV (R$)", "seller_hhi": "Seller HHI"},
)

portfolio_figure.update_layout(template="plotly_white", height=580)
portfolio_figure.show()


$$
\mathrm{HHI}_c = \sum_{s \in c} \left(\frac{\mathrm{GMV}_{s,c}}{\mathrm{GMV}_c}\right)^2
$$

Here, \(s\) is a seller in category \(c\). A value near 1 indicates that one seller dominates delivered-item GMV; a lower value indicates a more distributed seller mix within Olist.

**This chart compares seller counts and unique product counts across categories and uses the HHI formula above to measure seller concentration. A lower HHI indicates a more dispersed seller structure and lower monopoly risk within Olist. Computers has the highest concentration, with an HHI of about 0.58, so this category should generally be avoided when selecting products for market entry. Other relatively concentrated categories include office furniture, home comfort, and small appliances for ovens and coffee. Many categories on the right side of the chart have more sellers and lower HHI. Watches and gifts, computer accessories, bed and bath, housewares, sports and leisure, health and beauty, and auto are notable potential entry categories.**

## 7. Customer Analysis


### 7.1 Customer Base and Purchase Behavior


In [18]:
# Build a delivered-order customer table at one row per order
order_payments = payments_df.groupby("order_id", as_index=False)["payment_value"].sum()
delivered_customer_orders = (
    delivered_order_details.merge(customers_df[["customer_id", "customer_unique_id"]], on="customer_id", how="left")
    .merge(order_payments, on="order_id", how="left")
    .dropna(subset=["customer_unique_id"])
)
customer_activity = (
    delivered_customer_orders.groupby("customer_unique_id", as_index=False)
    .agg(order_count=("order_id", "nunique"), total_spend=("payment_value", "sum"), first_purchase=("order_purchase_timestamp", "min"), last_purchase=("order_purchase_timestamp", "max"))
)
customer_activity["repeat_customer"] = customer_activity["order_count"].gt(1)
frequency_thresholds = [1, 2, 3, 4, 5]
frequency_reach = pd.DataFrame({
    "stage": [f"At least {threshold} delivered order{'s' if threshold > 1 else ''}" for threshold in frequency_thresholds],
    "consumers": [(customer_activity["order_count"] >= threshold).sum() for threshold in frequency_thresholds],
})
frequency_reach["share_of_buyers"] = frequency_reach["consumers"] / frequency_reach["consumers"].iloc[0]
frequency_reach["log_consumers"] = np.log10(frequency_reach["consumers"] + 1)
frequency_figure = go.Figure(go.Funnel(
    y=frequency_reach["stage"], x=frequency_reach["log_consumers"],
    customdata=frequency_reach[["consumers", "share_of_buyers"]].to_numpy(),
    texttemplate="%{customdata[0]:,.0f} consumers · %{customdata[1]:.1%} of buyers",
    textposition="inside", marker={"color": ["#1D4ED8", "#2563EB", "#3B82F6", "#60A5FA", "#93C5FD"]},
    hovertemplate="%{y}<br>%{customdata[0]:,.0f} consumers<br>%{customdata[1]:.2%} of buyers<extra></extra>",
))
frequency_figure.update_layout(title="Delivered-Order Reach by Purchase Threshold", template="plotly_white", height=520, margin={"l": 210, "r": 40, "t": 80, "b": 70})
frequency_figure.update_xaxes(title="log10(Consumers + 1)")
frequency_figure.show()


**The funnel shows that more than 93,000 customers placed at least one delivered order on Olist, but only 3.00% placed a second or later order. Only 0.05% placed more than three orders, and the counts at higher purchase thresholds fall to the tens. This indicates very weak customer stickiness and a fragile core customer base, limiting the platform's ability to consolidate its position in online shopping. Developing a core group of repeat customers should therefore be a central goal for Olist's future growth.**

### 7.2 RFM Segmentation and Repeat Purchasing


In [19]:
# Create RFM scores from delivered-order customer activity
rfm_reference_date = delivered_customer_orders["order_purchase_timestamp"].max().normalize() + pd.Timedelta(days=1)
rfm = customer_activity[["customer_unique_id", "order_count", "total_spend", "last_purchase"]].copy()
rfm["recency_days"] = (rfm_reference_date - rfm["last_purchase"]).dt.days
rfm["R"] = pd.qcut(rfm["recency_days"].rank(method="first"), 4, labels=[4, 3, 2, 1]).astype(int)
rfm["F"] = pd.cut(rfm["order_count"], bins=[0, 1, 2, 3, np.inf], labels=[1, 2, 3, 4]).astype(int)
rfm["M"] = pd.qcut(rfm["total_spend"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
rfm["segment"] = np.select(
    [
        rfm["R"].ge(3) & rfm["F"].ge(3) & rfm["M"].ge(3),
        rfm["R"].ge(2) & rfm["F"].ge(2),
        rfm["R"].ge(3) & rfm["F"].eq(1),
        rfm["R"].le(2) & rfm["F"].ge(2),
    ],
    ["Champions", "Loyal Customers", "Promising One-Time Buyers", "At-Risk Repeat Buyers"],
    default="Other One-Time Buyers",
)


In [20]:
rfm_quadrant = (
    rfm.groupby("segment", as_index=False)
    .agg(customers=("customer_unique_id", "nunique"), average_recency_days=("recency_days", "mean"), average_orders=("order_count", "mean"), average_spend=("total_spend", "mean"))
)
rfm_quadrant["customer_share"] = rfm_quadrant["customers"] / rfm_quadrant["customers"].sum()
rfm_short_labels = {"Champions": "Champions", "Loyal Customers": "Loyal", "Promising One-Time Buyers": "Promising One-Time", "At-Risk Repeat Buyers": "At-Risk Repeat", "Other One-Time Buyers": "Other One-Time"}
rfm_quadrant["bubble_label"] = [f"{rfm_short_labels[segment]}<br>{customers:,} ({share:.1%})" for segment, customers, share in zip(rfm_quadrant["segment"], rfm_quadrant["customers"], rfm_quadrant["customer_share"])]
rfm_recency_cutoff = rfm["recency_days"].median()
rfm_repeat_cutoff = 1.5
rfm_quadrant_figure = px.scatter(
    rfm_quadrant, x="average_recency_days", y="average_orders",
    size="customers", color="segment", text="bubble_label", size_max=58,
    color_discrete_map={"Champions": "#0F766E", "Loyal Customers": "#2563EB", "Promising One-Time Buyers": "#F59E0B", "At-Risk Repeat Buyers": "#EA580C", "Other One-Time Buyers": "#64748B"},
    hover_data={"customers": ":,", "customer_share": ":.1%", "average_spend": ":.2f", "average_recency_days": ":.1f", "average_orders": ":.2f", "bubble_label": False},
    title="RFM Segment Quadrant: Recency vs Purchase Frequency",
    labels={"average_recency_days": "Average Days Since Last Purchase (lower is more recent)", "average_orders": "Average Delivered Orders per Consumer"},
)
rfm_quadrant_figure.add_vline(x=rfm_recency_cutoff, line_dash="dash", line_color="#94A3B8")
rfm_quadrant_figure.add_hline(y=rfm_repeat_cutoff, line_dash="dash", line_color="#94A3B8")
rfm_quadrant_figure.update_traces(textposition="top center", textfont_size=12, marker_line_width=1, marker_line_color="white")
rfm_quadrant_figure.update_layout(template="plotly_white", height=650, showlegend=False, margin={"l": 80, "r": 95, "t": 90, "b": 80})
rfm_quadrant_figure.show()

**Using RFM customer segmentation, the platform's buyers are divided into five groups: Champions, Loyal Customers, Promising One-Time Buyers, At-Risk Repeat Buyers, and Other One-Time Buyers. Promising One-Time Buyers represent 48.35% of customers, while Loyal Customers account for only 2.23% and Champions for 0.15%. This further shows that Olist urgently needs to convert its large Promising One-Time Buyer segment. At-Risk Repeat Buyers and Other One-Time Buyers can be given lower priority; the key is to retain and convert the large Promising One-Time Buyer group.**

### 7.3 Customer Value and Consumption Level


In [21]:
# Segment consumers by total delivered-order spend percentiles
customer_value = customer_activity.copy()
customer_value["value_tier"] = pd.qcut(
    customer_value["total_spend"].rank(method="first"),
    q=[0, 0.50, 0.80, 0.95, 1.00],
    labels=["Low Value", "Mid Value", "High Value", "Premium"],
)
value_tier_summary = (
    customer_value.groupby("value_tier", observed=False, as_index=False)
    .agg(customers=("customer_unique_id", "nunique"), spend=("total_spend", "sum"), average_orders=("order_count", "mean"))
)
value_tier_summary["customer_share"] = value_tier_summary["customers"] / value_tier_summary["customers"].sum()
value_tier_summary["spend_share"] = value_tier_summary["spend"] / value_tier_summary["spend"].sum()
value_tier_summary["spend_to_customer_ratio"] = value_tier_summary["spend_share"] / value_tier_summary["customer_share"]
value_figure = make_subplots(specs=[[{"secondary_y": True}]])
value_figure.add_trace(go.Bar(x=value_tier_summary["value_tier"], y=value_tier_summary["customer_share"], name="Customer Share", marker_color="#93C5FD", text=value_tier_summary["customer_share"], texttemplate="%{text:.1%}", textposition="outside"), secondary_y=False)
value_figure.add_trace(go.Bar(x=value_tier_summary["value_tier"], y=value_tier_summary["spend_share"], name="Spend Share", marker_color="#2563EB", text=value_tier_summary["spend_share"], texttemplate="%{text:.1%}", textposition="outside"), secondary_y=False)
value_figure.add_trace(go.Scatter(x=value_tier_summary["value_tier"], y=value_tier_summary["spend_to_customer_ratio"], name="Spend Share / Customer Share", mode="lines+markers+text", line={"color": "#EA580C", "width": 3}, marker={"size": 9}, text=[f"{ratio:.2f}×" for ratio in value_tier_summary["spend_to_customer_ratio"]], textposition="top center"), secondary_y=True)
value_figure.update_yaxes(title_text="Share of Total", tickformat=".0%", range=[0, value_tier_summary[["customer_share", "spend_share"]].max().max() * 1.25], secondary_y=False)
value_figure.update_yaxes(title_text="Spend Share / Customer Share (×)", rangemode="tozero", secondary_y=True)
value_figure.update_layout(title="Customer Value Tier Composition and Share Ratio", template="plotly_white", height=550, barmode="group", xaxis_title="Customer Value Tier", legend={"orientation": "h", "y": 1.12, "x": 0.5, "xanchor": "center"}, margin={"l": 70, "r": 85, "t": 120, "b": 60})
value_figure.show()


$$Q_t = \frac{S_t}{C_t}$$

\(C_t\) is the share of active consumers in value tier \(t\), and \(S_t\) is that tier's share of total spending. \(Q_t>1\) means its spending share exceeds its customer share.

**Low Value customers account for 50% of buyers and 19.2% of spending; Mid Value customers account for 30% and 27.2%; High Value customers account for 15% and 26.7%; and Premium customers account for 5% and 26.8%. The first two groups contain 80% of customers, while the latter two contain 20%. Comparing spending share with customer share shows that average spending per customer is approximately 2.36 times higher in Mid Value than Low Value, 1.96 times higher in High Value than Mid Value, and 3.01 times higher in Premium than High Value. The most reasonable conversion strategy is to move Low Value customers toward Mid Value while retaining the top 20% of customers, who contribute approximately 53.5% of recorded spending.**

In [22]:
value_tier_repeat = (
    customer_value.groupby("value_tier", observed=False, as_index=False)
    .agg(repeat_rate=("repeat_customer", "mean"))
)
value_quadrant = value_tier_summary.merge(value_tier_repeat, on="value_tier")
value_quadrant["average_spend"] = value_quadrant["spend"] / value_quadrant["customers"]
value_quadrant["customer_share"] = value_quadrant["customers"] / value_quadrant["customers"].sum()
value_quadrant["bubble_label"] = [f"{tier}<br>{customers:,} ({share:.1%})" for tier, customers, share in zip(value_quadrant["value_tier"], value_quadrant["customers"], value_quadrant["customer_share"])]
value_spend_cutoff = customer_value["total_spend"].mean()
value_repeat_cutoff = customer_value["repeat_customer"].mean()
value_quadrant_figure = px.scatter(
    value_quadrant, x="average_spend", y="repeat_rate",
    size="customers", color="value_tier", text="bubble_label", size_max=58,
    color_discrete_map={"Low Value": "#93C5FD", "Mid Value": "#2563EB", "High Value": "#7C3AED", "Premium": "#EA580C"},
    hover_data={"customers": ":,", "customer_share": ":.1%", "average_orders": ":.2f", "average_spend": ":.2f", "repeat_rate": ":.2%", "bubble_label": False},
    title="Customer Value Quadrant: Spending vs Repeat Purchase",
    labels={"average_spend": "Average Delivered-Order Spend per Consumer (R$)", "repeat_rate": "Repeat Purchase Rate", "value_tier": "Customer Value Tier"},
)
value_quadrant_figure.add_vline(x=value_spend_cutoff, line_dash="dash", line_color="#94A3B8")
value_quadrant_figure.add_hline(y=value_repeat_cutoff, line_dash="dash", line_color="#94A3B8")
value_quadrant_figure.update_traces(textposition="top center", textfont_size=12, marker_line_width=1, marker_line_color="white")
value_quadrant_figure.update_yaxes(tickformat=".0%")
value_quadrant_figure.update_layout(template="plotly_white", height=630, showlegend=False, margin={"l": 75, "r": 95, "t": 90, "b": 80})
value_quadrant_figure.show()

**The quadrant chart displays the same four customer-value groups. Because the Low Value and Mid Value bubbles are closest to each other, moving Low Value customers toward Mid Value appears to be a reasonable conversion strategy. Mid Value customers also occupy a potentially risky position near the intersection of the reference lines and close to the other quadrants. Olist should therefore work to increase their repeat-purchase rate and spending so that they do not move toward less favorable groups.**

## 8. Regional Market Analysis

### 8.1 Regional Demand and Customer Distribution

The offline Brazil outline is stored in source_data/brazil_boundary.geojson and derived from [Natural Earth 1:110m Admin 0](https://www.naturalearthdata.com/downloads/110m-cultural-vectors/). State points use median coordinates from the Olist geolocation file.

In [23]:
order_item_values = (
    items_df.groupby("order_id", as_index=False)
    .agg(gmv=("price", "sum"), freight_value=("freight_value", "sum"))
)

regional_orders = (
    delivered_order_details.merge(order_item_values, on="order_id", how="inner")
    .merge(
        customers_df[["customer_id", "customer_unique_id", "customer_state", "customer_city"]],
        on="customer_id",
        how="left",
    )
)

state_summary = (
    regional_orders.groupby("customer_state", as_index=False)
    .agg(
        active_customers=("customer_unique_id", "nunique"),
        delivered_orders=("order_id", "nunique"),
        gmv=("gmv", "sum"),
        freight_value=("freight_value", "sum"),
        avg_delivery_days=("delivery_days", "mean"),
        late_delivery_rate=("is_late_delivery", "mean"),
    )
)
state_summary["aov"] = state_summary["gmv"] / state_summary["delivered_orders"]
state_summary["freight_to_gmv"] = state_summary["freight_value"] / state_summary["gmv"]

state_demand_view = state_summary.nlargest(15, "gmv").sort_values("gmv")

In [24]:
# Build an offline geographic point map using local geolocation and Natural Earth data.
import json
with (DATA_DIR / "brazil_boundary.geojson").open(encoding="utf-8") as boundary_file:
    brazil_boundary = json.load(boundary_file)["geometry"]["coordinates"][0]
brazil_outline_lon, brazil_outline_lat = zip(*brazil_boundary)
valid_geo = geolocation_df.loc[
    geolocation_df["geolocation_lat"].between(-34.5, 5.5)
    & geolocation_df["geolocation_lng"].between(-74, -34)
].copy()
zip_locations = (
    valid_geo.groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(latitude=("geolocation_lat", "median"), longitude=("geolocation_lng", "median"))
)
background_locations = zip_locations.sample(n=min(5000, len(zip_locations)), random_state=42)
state_locations = (
    valid_geo.groupby("geolocation_state", as_index=False)
    .agg(latitude=("geolocation_lat", "median"), longitude=("geolocation_lng", "median"))
)
state_map = state_summary.merge(state_locations, left_on="customer_state", right_on="geolocation_state", how="inner")
state_map["marker_size"] = 12 + 28 * np.sqrt(state_map["gmv"] / state_map["gmv"].max())

state_map["gmv_millions"] = state_map["gmv"] / 1_000_000
top_map_states = set(state_map.nlargest(5, "gmv")["customer_state"])
state_map["map_label"] = state_map["customer_state"].where(state_map["customer_state"].isin(top_map_states), "")
regional_demand_figure = make_subplots(
    rows=1, cols=2, specs=[[{"type": "xy"}, {"type": "map"}]],
    column_widths=[0.43, 0.57], horizontal_spacing=0.07,
    subplot_titles=("Top States by GMV", "Geographic GMV Distribution"),
)
regional_demand_figure.add_trace(go.Bar(
    x=state_demand_view["gmv"] / 1_000_000, y=state_demand_view["customer_state"],
    orientation="h", marker_color="#2563EB", text=state_demand_view["gmv"] / 1_000_000,
    texttemplate="%{text:.2f}M", textposition="outside", cliponaxis=False,
    customdata=state_demand_view[["active_customers", "delivered_orders"]].to_numpy(),
    hovertemplate="<b>%{y}</b><br>GMV: R$ %{x:.2f}M<br>Active consumers: %{customdata[0]:,.0f}<br>Delivered orders: %{customdata[1]:,.0f}<extra></extra>",
    showlegend=False,
), row=1, col=1)
regional_demand_figure.add_trace(go.Scattermap(
    lat=background_locations["latitude"], lon=background_locations["longitude"],
    mode="markers", marker={"size": 3, "color": "#CBD5E1", "opacity": 0.45},
    hoverinfo="skip", showlegend=False,
), row=1, col=2)
regional_demand_figure.add_trace(go.Scattermap(
    lat=brazil_outline_lat, lon=brazil_outline_lon, mode="lines",
    line={"color": "#334155", "width": 2}, hoverinfo="skip", showlegend=False,
), row=1, col=2)
regional_demand_figure.add_trace(go.Scattermap(
    lat=state_map["latitude"], lon=state_map["longitude"],
    mode="markers+text", text=state_map["map_label"], hovertext=state_map["customer_state"], textposition="top center",
    marker={"size": state_map["marker_size"], "color": state_map["gmv_millions"], "colorscale": "Blues", "showscale": True, "colorbar": {"title": "R$ M", "x": 1.02, "len": 0.68}},
    customdata=state_map[["gmv", "active_customers", "delivered_orders", "late_delivery_rate"]].to_numpy(),
    hovertemplate="<b>%{hovertext}</b><br>GMV: R$ %{customdata[0]:,.0f}<br>Active consumers: %{customdata[1]:,.0f}<br>Delivered orders: %{customdata[2]:,.0f}<br>Late delivery: %{customdata[3]:.1%}<extra></extra>",
    showlegend=False,
), row=1, col=2)
regional_demand_figure.update_xaxes(title_text="GMV (R$ millions)", rangemode="tozero", row=1, col=1)
regional_demand_figure.update_layout(
    title="Delivered-Order GMV by Customer State",
    map={"style": "white-bg", "center": {"lat": -15, "lon": -52}, "zoom": 3.1},
    template="plotly_white", height=680, margin={"l": 65, "r": 100, "t": 95, "b": 55},
)
regional_demand_figure.show()

### 8.2 State and City Market Comparison

In [25]:
city_summary = (
    regional_orders.groupby(["customer_state", "customer_city"], as_index=False)
    .agg(
        active_customers=("customer_unique_id", "nunique"),
        delivered_orders=("order_id", "nunique"),
        gmv=("gmv", "sum"),
    )
)
city_summary["aov"] = city_summary["gmv"] / city_summary["delivered_orders"]
city_summary["city_state"] = city_summary["customer_city"].str.title() + " (" + city_summary["customer_state"] + ")"
city_market_view = city_summary.query("delivered_orders >= 100").nlargest(10, "gmv").sort_values("gmv")
state_market_view = state_summary.nlargest(10, "gmv").sort_values("gmv")

regional_comparison_figure = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Top States by GMV", "Top Cities by GMV (100+ Orders)"),
    horizontal_spacing=0.22,
)
regional_comparison_figure.add_trace(
    go.Bar(x=state_market_view["gmv"], y=state_market_view["customer_state"], orientation="h", marker_color="#2563EB", text=state_market_view["gmv"], texttemplate="R$ %{text:,.0f}"),
    row=1, col=1,
)
regional_comparison_figure.add_trace(
    go.Bar(x=city_market_view["gmv"], y=city_market_view["city_state"], orientation="h", marker_color="#0F9D78", text=city_market_view["gmv"], texttemplate="R$ %{text:,.0f}"),
    row=1, col=2,
)
regional_comparison_figure.update_layout(title="State and City Market Comparison", template="plotly_white", height=560, showlegend=False)
regional_comparison_figure.update_xaxes(title_text="GMV (R$)")
regional_comparison_figure.show()

### 8.3 Geographic Opportunity and Logistics Coverage

In [26]:
seller_state_coverage = (
    items_df[["order_id", "seller_id"]]
    .merge(delivered_order_details[["order_id", "customer_id"]], on="order_id", how="inner")
    .merge(customers_df[["customer_id", "customer_state"]], on="customer_id", how="left")
    .groupby("customer_state", as_index=False)
    .agg(active_sellers=("seller_id", "nunique"))
)
regional_opportunity = state_summary.merge(seller_state_coverage, on="customer_state", how="left")

regional_map = regional_opportunity.merge(state_locations, left_on="customer_state", right_on="geolocation_state", how="inner")
regional_map["marker_size"] = 12 + 38 * np.sqrt(regional_map["active_customers"] / regional_map["active_customers"].max())
top_opportunity_states = set(regional_map.nlargest(5, "gmv")["customer_state"])
regional_map["map_label"] = regional_map["customer_state"].where(regional_map["customer_state"].isin(top_opportunity_states), "")
delivery_rate_bounds = [regional_map["late_delivery_rate"].min(), regional_map["late_delivery_rate"].max()]
regional_opportunity_figure = make_subplots(
    rows=1, cols=2, specs=[[{"type": "xy"}, {"type": "map"}]],
    column_widths=[0.48, 0.52], horizontal_spacing=0.07,
    subplot_titles=("Demand vs. Seller Coverage", "Geographic Demand and Delivery"),
)
regional_opportunity_figure.add_trace(go.Scatter(
    x=regional_map["active_sellers"], y=regional_map["gmv"],
    mode="markers+text", text=regional_map["customer_state"], textposition="top center",
    marker={"size": regional_map["marker_size"], "color": regional_map["late_delivery_rate"],
            "colorscale": "RdYlGn_r", "cmin": delivery_rate_bounds[0], "cmax": delivery_rate_bounds[1],
            "showscale": False, "opacity": 0.8},
    customdata=regional_map[["customer_state", "active_customers", "late_delivery_rate"]].to_numpy(),
    hovertemplate="<b>%{customdata[0]}</b><br>Active sellers: %{x:,.0f}<br>GMV: R$ %{y:,.0f}<br>Active consumers: %{customdata[1]:,.0f}<br>Late delivery: %{customdata[2]:.1%}<extra></extra>",
    showlegend=False,
), row=1, col=1)
regional_opportunity_figure.add_trace(go.Scattermap(
    lat=background_locations["latitude"], lon=background_locations["longitude"],
    mode="markers", marker={"size": 3, "color": "#CBD5E1", "opacity": 0.45},
    hoverinfo="skip", showlegend=False,
), row=1, col=2)
regional_opportunity_figure.add_trace(go.Scattermap(
    lat=brazil_outline_lat, lon=brazil_outline_lon, mode="lines",
    line={"color": "#334155", "width": 2}, hoverinfo="skip", showlegend=False,
), row=1, col=2)
regional_opportunity_figure.add_trace(go.Scattermap(
    lat=regional_map["latitude"], lon=regional_map["longitude"],
    mode="markers+text", text=regional_map["map_label"], hovertext=regional_map["customer_state"], textposition="top center",
    marker={"size": regional_map["marker_size"], "color": regional_map["late_delivery_rate"],
            "colorscale": "RdYlGn_r", "cmin": delivery_rate_bounds[0], "cmax": delivery_rate_bounds[1],
            "showscale": True, "colorbar": {"title": "Late Rate", "tickformat": ".0%", "x": 1.02, "len": 0.68}},
    customdata=regional_map[["gmv", "active_customers", "active_sellers", "late_delivery_rate"]].to_numpy(),
    hovertemplate="<b>%{hovertext}</b><br>GMV: R$ %{customdata[0]:,.0f}<br>Active consumers: %{customdata[1]:,.0f}<br>Active sellers: %{customdata[2]:,.0f}<br>Late delivery: %{customdata[3]:.1%}<extra></extra>",
    showlegend=False,
), row=1, col=2)
regional_opportunity_figure.update_xaxes(title_text="Active Sellers Serving State", rangemode="tozero", row=1, col=1)
regional_opportunity_figure.update_yaxes(title_text="GMV (R$)", rangemode="tozero", row=1, col=1)
regional_opportunity_figure.update_layout(
    title="Regional Demand and Logistics Coverage",
    map={"style": "white-bg", "center": {"lat": -15, "lon": -52}, "zoom": 3.1},
    template="plotly_white", height=680, margin={"l": 75, "r": 100, "t": 95, "b": 65},
)
regional_opportunity_figure.show()

## 9. Delivery and Logistics Analysis

### 9.1 Delivery Time, Timeliness, and Delays

In [27]:
# Find the first delivery day whose mean review score falls below the overall mean.
delivery_performance = delivered_order_details.loc[
    delivered_order_details["delivery_days"].notna()
    & delivered_order_details["delivery_days"].ge(0)
    & delivered_order_details["delivery_delay_days"].notna()
].copy()
on_time_rate = 1 - delivery_performance["is_late_delivery"].mean()
latest_delivery_reviews = (
    reviews_df.dropna(subset=["review_score"])
    .sort_values("review_answer_timestamp", na_position="first")
    .drop_duplicates(subset="order_id", keep="last")
)
delivery_review_days = delivery_performance[["order_id", "delivery_days"]].merge(
    latest_delivery_reviews[["order_id", "review_score"]], on="order_id", how="inner"
)
overall_delivered_review_mean = delivery_review_days["review_score"].mean()
delivery_time_limit = delivery_review_days["delivery_days"].quantile(0.99)
delivery_review_days = delivery_review_days.loc[delivery_review_days["delivery_days"].le(delivery_time_limit)].copy()
delivery_review_days["delivery_day"] = np.floor(delivery_review_days["delivery_days"]).astype(int)
delivery_review_days["low_review"] = delivery_review_days["review_score"].le(2)
delivery_review_days["high_review"] = delivery_review_days["review_score"].ge(4)
delivery_day_summary = (
    delivery_review_days.groupby("delivery_day", as_index=False)
    .agg(
        reviewed_orders=("order_id", "nunique"),
        low_review_rate=("low_review", "mean"),
        high_review_rate=("high_review", "mean"),
        average_review_score=("review_score", "mean"),
    )
)
delivery_day_summary = delivery_day_summary.loc[delivery_day_summary["reviewed_orders"].ge(30)]
below_average_days = delivery_day_summary.loc[
    delivery_day_summary["average_review_score"].lt(overall_delivered_review_mean)
]
review_below_average_day = int(below_average_days["delivery_day"].iloc[0]) if not below_average_days.empty else None

delivery_review_figure = make_subplots(
    rows=1, cols=2, column_widths=[0.54, 0.46], horizontal_spacing=0.11,
    subplot_titles=("Mean Review vs. Overall Average", "Review Volume and Mean Score"),
)
delivery_review_figure.add_trace(go.Scatter(
    x=delivery_day_summary["delivery_day"], y=delivery_day_summary["average_review_score"],
    mode="lines+markers", name="Daily mean review",
    line={"color": "#2563EB", "width": 2.5}, marker={"size": 5},
), row=1, col=1)
delivery_review_figure.add_hline(
    y=overall_delivered_review_mean, line_color="#64748B", line_dash="dash",
    annotation_text=f"Overall mean: {overall_delivered_review_mean:.2f}", row=1, col=1,
)
delivery_review_figure.add_trace(go.Bar(
    x=delivery_day_summary["delivery_day"], y=delivery_day_summary["reviewed_orders"],
    marker={
        "color": delivery_day_summary["average_review_score"], "colorscale": "RdYlGn",
        "cmin": 1, "cmax": 5, "showscale": True,
        "colorbar": {"title": "Mean Review", "x": 1.02, "len": 0.75},
    },
    customdata=delivery_day_summary[["average_review_score", "low_review_rate", "high_review_rate"]].to_numpy(),
    hovertemplate="Delivery: %{x} days<br>Reviewed orders: %{y:,.0f}<br>Mean review: %{customdata[0]:.2f} / 5<br>Low score: %{customdata[1]:.1%}<br>High score: %{customdata[2]:.1%}<extra></extra>",
    showlegend=False,
), row=1, col=2)
if review_below_average_day is not None:
    for column in [1, 2]:
        delivery_review_figure.add_vline(
            x=review_below_average_day, line_color="#7C3AED", line_dash="dash", line_width=2,
            annotation_text=f"First below mean: {review_below_average_day} days" if column == 1 else None,
            row=1, col=column,
        )
delivery_review_figure.update_xaxes(title_text="Purchase-to-Delivery Days")
delivery_review_figure.update_yaxes(title_text="Mean Review Score (1-5)", range=[1, 5], row=1, col=1)
delivery_review_figure.update_yaxes(title_text="Reviewed Orders", row=1, col=2)
delivery_review_figure.update_layout(
    title="Review Outcomes by Delivery Time (First 99% of Durations; 30+ Reviews per Day)",
    template="plotly_white", height=550,
    legend={"orientation": "h", "y": 1.13}, margin={"l": 70, "r": 100, "t": 110, "b": 65},
)
delivery_review_figure.show()
print(
    f"First delivery day with mean review below the overall mean: {review_below_average_day} days"
    if review_below_average_day is not None else "No reviewed delivery-day bin falls below the overall mean."
)

First delivery day with mean review below the overall mean: 16 days


### 9.2 Freight Cost and Delivery Economics

In [28]:
freight_category_view = category_economics.loc[category_economics["items"].ge(100)].copy()

### 9.3 Order Status and Delivery Exceptions

In [29]:
order_status_summary = (
    orders_df.groupby("order_status", as_index=False)
    .agg(orders=("order_id", "nunique"))
    .sort_values("orders", ascending=False)
)
order_status_summary["order_share"] = order_status_summary["orders"] / order_status_summary["orders"].sum()
delivered_count = int(order_status_summary.loc[order_status_summary["order_status"].eq("delivered"), "orders"].sum())
other_count = int(order_status_summary["orders"].sum() - delivered_count)
other_share = other_count / (delivered_count + other_count)
other_slice_angle = 360 * other_share
other_statuses = order_status_summary.loc[order_status_summary["order_status"].ne("delivered")].copy()
status_colors = {"delivered": "#2563EB", "shipped": "#0F766E", "canceled": "#EA580C", "unavailable": "#DC2626", "invoiced": "#7C3AED", "processing": "#F59E0B", "approved": "#64748B", "created": "#94A3B8"}
order_status_figure = go.Figure()
order_status_figure.add_trace(go.Pie(
    labels=["Other statuses", "Delivered"], values=[other_count, delivered_count],
    hole=0.62, domain={"x": [0.00, 0.40], "y": [0.12, 0.90]},
    marker={"colors": ["#EA580C", status_colors["delivered"]]},
    textinfo="percent", textposition="inside", sort=False, direction="clockwise", rotation=90 - other_slice_angle / 2, showlegend=False,
    hovertemplate="%{label}: %{value:,.0f} orders (%{percent})<extra></extra>",
))
order_status_figure.add_trace(go.Pie(
    labels=other_statuses["order_status"].str.title(), values=other_statuses["orders"],
    hole=0.62, domain={"x": [0.60, 1.00], "y": [0.12, 0.90]},
    marker={"colors": [status_colors.get(status, "#94A3B8") for status in other_statuses["order_status"]]},
    textinfo="label+percent", textposition="outside", sort=False, showlegend=False,
    hovertemplate="%{label}: %{value:,.0f} orders (%{percent} of non-delivered)<extra></extra>",
))
order_status_figure.update_layout(
    title="Order Outcomes and Non-Delivered Status Breakdown", template="plotly_white", height=550,
    annotations=[
        {"x": 0.20, "y": 0.51, "xref": "paper", "yref": "paper", "text": f"All orders<br>{delivered_count + other_count:,}", "showarrow": False, "font": {"size": 16}},
        {"x": 0.80, "y": 0.51, "xref": "paper", "yref": "paper", "text": f"Non-delivered<br>{other_count:,}", "showarrow": False, "font": {"size": 16}},
        {"x": 0.50, "y": 0.62, "xref": "paper", "yref": "paper", "text": f"Other statuses<br>{other_share:.1%} of all orders", "showarrow": False, "font": {"size": 13, "color": "#EA580C"}},
        {"x": 0.60, "y": 0.51, "ax": -20, "ay": 0, "xref": "paper", "yref": "paper", "text": " ", "showarrow": True, "arrowhead": 3, "arrowsize": 1.2, "arrowwidth": 2.5, "arrowcolor": "#EA580C"},
    ],
    shapes=[{"type": "line", "xref": "paper", "yref": "paper", "x0": 0.40, "x1": 0.58, "y0": 0.51, "y1": 0.51, "line": {"color": "#EA580C", "width": 2.5}}],
    margin={"l": 50, "r": 70, "t": 90, "b": 40},
)
order_status_figure.show()

The left ring shows delivered versus all other orders. The connected right ring breaks down only the non-delivered group, so its percentages use non-delivered orders as the denominator.

## 10. Customer Experience and Review Analysis

### 10.1 Review Score Distribution and Satisfaction Levels

In [30]:
order_reviews = (
    reviews_df.dropna(subset=["review_score"])
    .sort_values("review_answer_timestamp", na_position="first")
    .drop_duplicates(subset="order_id", keep="last")
)
review_distribution = (
    order_reviews.groupby("review_score", as_index=False)
    .agg(reviews=("order_id", "nunique"))
    .sort_values("review_score")
)
review_distribution["review_share"] = review_distribution["reviews"] / review_distribution["reviews"].sum()
average_review_score = order_reviews["review_score"].mean()

review_distribution_figure = px.bar(
    review_distribution,
    x="review_score",
    y="reviews",
    color="review_score",
    color_continuous_scale=["#D73027", "#FC8D59", "#FEE08B", "#91CF60", "#1A9850"],
    text="review_share",
    title=f"Review Score Distribution | Average Score: {average_review_score:.2f}",
    labels={"review_score": "Review Score", "reviews": "Reviewed Orders"},
)
review_distribution_figure.update_traces(texttemplate="%{text:.1%}", textposition="outside")
review_distribution_figure.update_layout(template="plotly_white", height=500, coloraxis_showscale=False)
review_distribution_figure.show()

### 10.2 Review Drivers and Low-Score Order Analysis

In [31]:
review_delivery = order_reviews[["order_id", "review_score"]].merge(
    delivery_performance[["order_id", "delivery_days", "delivery_delay_days", "is_late_delivery"]],
    on="order_id",
    how="inner",
)
review_delivery["delivery_status"] = np.where(review_delivery["is_late_delivery"], "Late", "On Time")
review_delivery["is_low_score"] = review_delivery["review_score"] <= 2
delivery_review_summary = (
    review_delivery.groupby("delivery_status", as_index=False)
    .agg(
        reviewed_orders=("order_id", "nunique"),
        average_review_score=("review_score", "mean"),
        low_score_rate=("is_low_score", "mean"),
    )
)

review_driver_figure = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Average Review Score", "Low-Score Review Rate"),
)
review_driver_figure.add_trace(
    go.Bar(x=delivery_review_summary["delivery_status"], y=delivery_review_summary["average_review_score"], marker_color=["#D73027", "#1A9850"], text=delivery_review_summary["average_review_score"], texttemplate="%{text:.2f}"),
    row=1, col=1,
)
review_driver_figure.add_trace(
    go.Bar(x=delivery_review_summary["delivery_status"], y=delivery_review_summary["low_score_rate"], marker_color=["#D73027", "#1A9850"], text=delivery_review_summary["low_score_rate"], texttemplate="%{text:.1%}"),
    row=1, col=2,
)
review_driver_figure.update_yaxes(title_text="Average Score", range=[0, 5], row=1, col=1)
review_driver_figure.update_yaxes(title_text="Low-Score Rate", tickformat=".0%", row=1, col=2)
review_driver_figure.update_layout(title="Review Outcomes by Delivery Timeliness", template="plotly_white", height=500, showlegend=False)
review_driver_figure.show()

### 10.3 Average Review Score by Customer State

Each delivered order contributes at most one review. All customer states are shown. Colors represent the difference from the reviewed-delivered-order mean in units of the review-count-weighted standard deviation of state means: green is above the mean, red is below it, and gray is near it. This standardized scale does not assume normality or establish statistical significance. Bubble size represents reviewed-order count.

In [32]:
# Rank customer states by mean review score and show their locations offline.
state_review_summary = (
    order_reviews[["order_id", "review_score"]]
    .merge(delivered_order_details[["order_id", "customer_id"]], on="order_id", how="inner")
    .merge(customers_df[["customer_id", "customer_state"]], on="customer_id", how="left")
    .dropna(subset=["customer_state"])
    .groupby("customer_state", as_index=False)
    .agg(reviewed_orders=("order_id", "nunique"), average_review_score=("review_score", "mean"))
)
national_review_mean = np.average(
    state_review_summary["average_review_score"], weights=state_review_summary["reviewed_orders"]
)
state_mean_sd = np.sqrt(np.average(
    (state_review_summary["average_review_score"] - national_review_mean) ** 2,
    weights=state_review_summary["reviewed_orders"],
))
state_review_summary["score_z"] = (
    (state_review_summary["average_review_score"] - national_review_mean) / state_mean_sd
)
state_review_rank = state_review_summary.sort_values("average_review_score")
state_review_map = state_review_summary.merge(
    state_locations, left_on="customer_state", right_on="geolocation_state", how="inner"
)
state_review_map["marker_size"] = 10 + 30 * np.sqrt(
    state_review_map["reviewed_orders"] / state_review_map["reviewed_orders"].max()
)
top_review_states = set(state_review_map.nlargest(5, "reviewed_orders")["customer_state"])
state_review_map["map_label"] = state_review_map["customer_state"].where(
    state_review_map["customer_state"].isin(top_review_states), ""
)
state_review_figure = make_subplots(
    rows=1, cols=2, specs=[[{"type": "xy"}, {"type": "map"}]],
    column_widths=[0.46, 0.54], horizontal_spacing=0.07,
    subplot_titles=("Mean Review Score Ranking", "Geographic Review Scores"),
)
state_review_figure.add_trace(go.Bar(
    x=state_review_rank["average_review_score"], y=state_review_rank["customer_state"],
    orientation="h", marker={"color": state_review_rank["score_z"],
        "colorscale": [[0, "#D73027"], [0.5, "#E5E7EB"], [1, "#1A9850"]],
        "cmin": -2, "cmax": 2},
    customdata=state_review_rank[["reviewed_orders", "score_z"]].to_numpy(),
    hovertemplate="<b>%{y}</b><br>Mean review: %{x:.2f} / 5<br>Difference: %{customdata[1]:+.2f} SD<br>Reviewed orders: %{customdata[0]:,.0f}<extra></extra>",
    showlegend=False,
), row=1, col=1)
state_review_figure.add_vline(
    x=national_review_mean, line_color="#475569", line_dash="dash",
    annotation_text=f"Brazil mean {national_review_mean:.2f}", row=1, col=1,
)
state_review_figure.add_trace(go.Scattermap(
    lat=brazil_outline_lat, lon=brazil_outline_lon, mode="lines",
    line={"color": "#334155", "width": 2}, hoverinfo="skip", showlegend=False,
), row=1, col=2)
state_review_figure.add_trace(go.Scattermap(
    lat=state_review_map["latitude"], lon=state_review_map["longitude"],
    mode="markers+text", text=state_review_map["map_label"],
    hovertext=state_review_map["customer_state"], textposition="top center",
    marker={
        "size": state_review_map["marker_size"],
        "color": state_review_map["score_z"],
        "colorscale": [[0, "#D73027"], [0.5, "#E5E7EB"], [1, "#1A9850"]],
        "cmin": -2, "cmax": 2,
        "showscale": True, "colorbar": {"title": "Difference", "x": 1.02, "len": 0.7,
            "tickvals": [-2, -1, 0, 1, 2],
            "ticktext": ["-2 SD", "-1 SD", "Mean", "+1 SD", "+2 SD"]},
    },
    customdata=state_review_map[["average_review_score", "reviewed_orders", "score_z"]].to_numpy(),
    hovertemplate="<b>%{hovertext}</b><br>Mean review: %{customdata[0]:.2f} / 5<br>Difference: %{customdata[2]:+.2f} SD<br>Reviewed orders: %{customdata[1]:,.0f}<extra></extra>",
    showlegend=False,
), row=1, col=2)
state_review_figure.update_xaxes(title_text="Mean Review Score (1-5)", range=[0, 5], row=1, col=1)
state_review_figure.update_layout(
    title=f"Customer-State Review Scores | Brazil Mean {national_review_mean:.2f}, 1 SD {state_mean_sd:.3f}",
    map={"style": "white-bg", "center": {"lat": -15, "lon": -52}, "zoom": 3.1},
    template="plotly_white", height=760, margin={"l": 75, "r": 105, "t": 95, "b": 55},
)
state_review_figure.show()

## 11. China-to-Brazil Market Entry Analysis

### 11.1 Regression Question and Category-Level Data

In [33]:
# Assemble one observation per category for an exploratory demand model.
order_category_pairs = product_sales[["order_id", "category_name"]].drop_duplicates()
category_reviews = (
    order_category_pairs.merge(order_reviews[["order_id", "review_score"]], on="order_id", how="inner")
    .groupby("category_name", as_index=False)
    .agg(reviewed_orders=("order_id", "nunique"), average_review_score=("review_score", "mean"))
)
category_model_data = (
    category_performance[["category_name", "orders", "gmv"]]
    .merge(category_economics[["category_name", "median_price", "freight_to_price_ratio"]], on="category_name")
    .merge(category_portfolio[["category_name", "unique_sellers", "seller_hhi"]], on="category_name")
    .merge(category_reviews, on="category_name")
)
category_model_data = category_model_data.loc[
    category_model_data["orders"].ge(100) & category_model_data["reviewed_orders"].ge(30)
].dropna().copy()
category_model_data["log_orders"] = np.log1p(category_model_data["orders"])
category_model_data["log_median_price"] = np.log1p(category_model_data["median_price"])
category_model_data["log_unique_sellers"] = np.log1p(category_model_data["unique_sellers"])
entry_model_features = [
    "log_median_price", "freight_to_price_ratio", "log_unique_sellers",
    "seller_hhi", "average_review_score",
]
assert len(category_model_data) > len(entry_model_features) + 10, "Too few categories for a holdout regression."
print(f"Model sample: {len(category_model_data)} categories, each with 100+ delivered orders and 30+ reviewed orders.")
display(category_model_data[["orders", "median_price", "freight_to_price_ratio", "unique_sellers", "seller_hhi", "average_review_score"]].describe().loc[["min", "50%", "max"]])

Model sample: 52 categories, each with 100+ delivered orders and 30+ reviewed orders.


,orders,median_price,freight_to_price_ratio,unique_sellers,seller_hhi,average_review_score
min,106.0,21.890,0.044199,9.0,0.012710,3.644695
50%,552.5,79.475,0.185649,55.5,0.091784,4.147874
max,9272.0,1100.000,0.365183,479.0,0.577193,4.534553


The dependent variable is category-level delivered-order demand. With standardized predictors \(z(\cdot)\), the exploratory model is

$$
\log(1+\mathrm{Orders}_c)=\beta_0+\beta_1 z\!\left(\log(1+\mathrm{MedianPrice}_c)\right)
+\beta_2 z(\mathrm{FreightRatio}_c)+\beta_3 z\!\left(\log(1+\mathrm{Sellers}_c)\right)
+\beta_4 z(\mathrm{SellerHHI}_c)+\beta_5 z(\mathrm{MeanReview}_c)+\varepsilon_c.
$$

Categories need at least 100 delivered orders and 30 reviewed orders. All predictors come from the same historical Olist period, so coefficients describe associations, not causal effects or an entry-success probability.

### 11.2 Model Fit and Coefficient Diagnostics

In [34]:
# Fit a standardized multivariable linear model and evaluate unseen categories.
random_indices = np.random.default_rng(42).permutation(len(category_model_data))
split_index = int(0.75 * len(random_indices))
train_indices, test_indices = random_indices[:split_index], random_indices[split_index:]
raw_features = category_model_data[entry_model_features].to_numpy(dtype=float)
target_log_orders = category_model_data["log_orders"].to_numpy(dtype=float)
feature_center = raw_features[train_indices].mean(axis=0)
feature_scale = raw_features[train_indices].std(axis=0, ddof=0)
feature_scale = np.where(feature_scale == 0, 1, feature_scale)
standardized_features = (raw_features - feature_center) / feature_scale
model_design = np.column_stack([np.ones(len(category_model_data)), standardized_features])
train_design, test_design = model_design[train_indices], model_design[test_indices]
train_target, test_target = target_log_orders[train_indices], target_log_orders[test_indices]
entry_coefficients = np.linalg.lstsq(train_design, train_target, rcond=None)[0]
train_fitted = train_design @ entry_coefficients
test_fitted = test_design @ entry_coefficients
test_r_squared = 1 - np.sum((test_target - test_fitted) ** 2) / np.sum((test_target - test_target.mean()) ** 2)
test_mae_orders = np.mean(np.abs(np.expm1(test_target) - np.maximum(np.expm1(test_fitted), 0)))

residual_degrees = len(train_target) - train_design.shape[1]
residual_variance = np.sum((train_target - train_fitted) ** 2) / residual_degrees
coefficient_standard_errors = np.sqrt(
    np.diag(residual_variance * np.linalg.pinv(train_design.T @ train_design))
)
coefficient_margins = student_t.ppf(0.975, residual_degrees) * coefficient_standard_errors
coefficient_summary = pd.DataFrame({
    "feature": entry_model_features,
    "standardized_coefficient": entry_coefficients[1:],
    "ci_low": entry_coefficients[1:] - coefficient_margins[1:],
    "ci_high": entry_coefficients[1:] + coefficient_margins[1:],
})
feature_labels = {
    "log_median_price": "Median price (log)",
    "freight_to_price_ratio": "Freight / price",
    "log_unique_sellers": "Seller count (log)",
    "seller_hhi": "Seller concentration",
    "average_review_score": "Mean review score",
}
coefficient_summary["label"] = coefficient_summary["feature"].map(feature_labels)
coefficient_summary = coefficient_summary.sort_values("standardized_coefficient")

entry_fit_figure = make_subplots(
    rows=1, cols=2, column_widths=[0.50, 0.50], horizontal_spacing=0.16,
    subplot_titles=("Held-Out Category Fit", "Standardized Coefficients and 95% CIs"),
)
entry_fit_figure.add_trace(go.Scatter(
    x=test_target, y=test_fitted, mode="markers",
    marker={"color": "#2563EB", "size": 10},
    hovertext=category_model_data.iloc[test_indices]["category_name"],
    hovertemplate="<b>%{hovertext}</b><br>Observed log orders: %{x:.2f}<br>Fitted log orders: %{y:.2f}<extra></extra>",
    showlegend=False,
), row=1, col=1)
reference_limits = [
    min(test_target.min(), test_fitted.min()),
    max(test_target.max(), test_fitted.max()),
]
entry_fit_figure.add_trace(go.Scatter(
    x=reference_limits, y=reference_limits, mode="lines",
    line={"color": "#64748B", "dash": "dash"}, hoverinfo="skip", showlegend=False,
), row=1, col=1)
entry_fit_figure.add_trace(go.Scatter(
    x=coefficient_summary["standardized_coefficient"], y=coefficient_summary["label"],
    mode="markers", marker={"color": "#EA580C", "size": 11},
    error_x={"type": "data", "array": coefficient_margins[1:][coefficient_summary.index], "visible": True},
    customdata=coefficient_summary[["ci_low", "ci_high"]].to_numpy(),
    hovertemplate="<b>%{y}</b><br>Coefficient: %{x:.2f}<br>95% CI: [%{customdata[0]:.2f}, %{customdata[1]:.2f}]<extra></extra>",
    showlegend=False,
), row=1, col=2)
entry_fit_figure.add_vline(x=0, line_color="#64748B", line_dash="dot", row=1, col=2)
entry_fit_figure.update_xaxes(title_text="Observed log(1 + orders)", row=1, col=1)
entry_fit_figure.update_yaxes(title_text="Fitted log(1 + orders)", row=1, col=1)
entry_fit_figure.update_xaxes(title_text="Coefficient per 1 SD Increase", row=1, col=2)
entry_fit_figure.update_layout(
    title=f"Exploratory Category Demand Model | Holdout R² = {test_r_squared:.2f}",
    template="plotly_white", height=570, margin={"l": 75, "r": 55, "t": 95, "b": 75},
)
entry_fit_figure.show()
display(pd.DataFrame([{
    "Training Categories": len(train_indices),
    "Holdout Categories": len(test_indices),
    "Holdout R²": round(test_r_squared, 3),
    "Holdout MAE (orders)": round(test_mae_orders),
}]))

,Training Categories,Holdout Categories,Holdout R²,Holdout MAE (orders)
0,39,13,0.845,866


The left panel compares observed and fitted log order counts for held-out categories; the diagonal represents perfect fit. The right panel shows standardized regression coefficients with model-based 95% confidence intervals. A coefficient interval crossing zero is not strong evidence of a stable association. The random category holdout does not test future-period performance.

### 11.3 Exploratory Category Screening

In [35]:
# Screen categories using fitted demand and observed operational proxies.
category_screen = category_model_data.copy()
category_screen["fitted_orders"] = np.maximum(np.expm1(model_design @ entry_coefficients), 0)
freight_median = category_screen["freight_to_price_ratio"].median()
review_median = category_screen["average_review_score"].median()
category_shortlist = (
    category_screen.loc[
        category_screen["freight_to_price_ratio"].le(freight_median)
        & category_screen["average_review_score"].ge(review_median)
    ]
    .nlargest(8, "fitted_orders")
    [["category_name", "fitted_orders", "orders", "gmv", "median_price",
      "freight_to_price_ratio", "seller_hhi", "average_review_score"]]
    .copy()
)
category_shortlist["category_name"] = category_shortlist["category_name"].str.replace("_", " ").str.title()
display(category_shortlist.style.format({
    "fitted_orders": "{:,.0f}", "orders": "{:,.0f}", "gmv": "R$ {:,.0f}",
    "median_price": "R$ {:,.2f}", "freight_to_price_ratio": "{:.1%}",
    "seller_hhi": "{:.2f}", "average_review_score": "{:.2f}",
}))
if test_r_squared <= 0:
    print("Warning: negative holdout R²; the shortlist is descriptive only and should not guide investment.")

,category_name,fitted_orders,orders,gmv,median_price,freight_to_price_ratio,seller_hhi,average_review_score
44,Health Beauty,"9,182","8,647","R$ 1,233,132",R$ 79.90,14.5%,0.02,4.23
68,Sports Leisure,"8,689","7,530","R$ 954,853",R$ 77.90,17.1%,0.02,4.23
21,Cool Stuff,"3,997","3,559","R$ 610,204",R$ 129.90,13.4%,0.07,4.22
72,Toys,"3,460","3,804","R$ 471,286",R$ 79.90,16.1%,0.04,4.24
61,Perfumery,"2,389","3,086","R$ 390,145",R$ 84.99,13.6%,0.08,4.26
62,Pet Shop,"1,306","1,688","R$ 211,696",R$ 89.70,18.4%,0.03,4.27
66,Small Appliances,875,609,"R$ 182,754",R$ 98.36,8.4%,0.07,4.25
17,Consoles Games,807,"1,018","R$ 148,003",R$ 54.89,12.9%,0.09,4.18


The shortlist combines model-fitted demand with below-median freight burden and above-median review score. It is a screening view, not a market-entry recommendation: domestic Olist transactions do not reveal import costs, regulations, Chinese product competitiveness, or platform-wide market share.

### 11.4 KPI Evidence and External Validation

The indicators below support an exploratory discussion of demand, competition, geographic concentration, logistics, and customer experience within Olist. They are not a market-entry success score. Any investment decision also needs current import duties, compliance requirements, cross-border logistics costs, and independent Brazilian market evidence.

In [36]:
# Keep service benchmarks relevant to operational readiness.
high_review_share = (order_reviews["review_score"] >= 4).mean()
service_benchmark_figure = make_subplots(
    rows=1, cols=2, specs=[[{"type": "indicator"}, {"type": "indicator"}]],
    horizontal_spacing=0.12,
)
for value, title, color, col in [
    (on_time_rate, "On-Time Delivery Rate", "#0F9D78", 1),
    (high_review_share, "High Review Score Share", "#F59E0B", 2),
]:
    service_benchmark_figure.add_trace(
        go.Indicator(
            mode="number", value=value,
            number={"valueformat": ".1%", "font": {"size": 42, "color": color}},
            title={"text": title, "font": {"size": 18}},
        ),
        row=1, col=col,
    )
service_benchmark_figure.update_layout(
    title="Observed Service Benchmarks from Olist Transactions",
    template="plotly_white", height=320, margin={"l": 35, "r": 35, "t": 75, "b": 30},
)
service_benchmark_figure.show()

entry_kpi_evidence = pd.DataFrame([
    {"Dimension": "Platform demand", "KPI": "Delivered-order GMV", "Olist Value": f"R$ {gmv:,.0f}", "Use in screening": "Observed transaction value, not Brazil-wide market size"},
    {"Dimension": "Platform demand", "KPI": "Delivered orders", "Olist Value": f"{order_volume:,.0f}", "Use in screening": "Observed transaction volume"},
    {"Dimension": "Competition", "KPI": "Active sellers", "Olist Value": f"{active_sellers:,.0f}", "Use in screening": "Platform seller participation, not all competitors"},
    {"Dimension": "Geography", "KPI": "Top-five state GMV share", "Olist Value": f"{state_summary.nlargest(5, 'gmv')['gmv'].sum() / state_summary['gmv'].sum():.1%}", "Use in screening": "Demand concentration across customer states"},
    {"Dimension": "Logistics", "KPI": "Charged freight / item GMV", "Olist Value": f"{delivered_items['freight_value'].sum() / delivered_items['price'].sum():.1%}", "Use in screening": "Observed domestic freight burden, not cross-border cost"},
    {"Dimension": "Logistics", "KPI": "On-time delivery rate", "Olist Value": f"{on_time_rate:.1%}", "Use in screening": "Domestic delivery-service benchmark"},
    {"Dimension": "Customer experience", "KPI": "High-review share (4-5)", "Olist Value": f"{high_review_share:.1%}", "Use in screening": "Share among reviewed orders only"},
])

## 12. Key Findings

These findings describe historical transactions on Olist, not the size or performance of the entire Brazilian e-commerce market.

1. **Platform scale and growth.** Delivered orders generated R$13.22 million in item-price GMV across 96,478 orders, or R$137.04 per delivered order. The dataset includes 93,358 transacting customers and 2,970 active sellers under these delivered-order definitions. Monthly GMV and delivered-order counts generally increased over the observed period, but the partial boundary years should not be compared as full years.
2. **Concentrated category demand.** About 24.3% of observed product categories account for 80% of delivered-item GMV. Category-level pricing, freight burden, seller count, and seller concentration vary materially, so category screening is more informative than applying one marketplace-wide assumption. This is an Olist share, not national market share.
3. **Uneven customer value and infrequent observed repeat buying.** In the spend-ranked delivered-order segmentation, the highest-spending 5% of customers contributed 26.8% of observed spend, while the lowest-spending 50% contributed 19.2%. Among consumers with defined payment records in 2017, 2.9% placed more than one observed order. The active-month spend average (R$163.51) and active-year spend average (R$165.85) use different denominators and should not be annualized against each other. The finite observation window limits retention inference.
4. **Geographic concentration and delivery exposure.** The top five customer states contributed 73.9% of delivered-item GMV. Charged domestic freight equaled 16.6% of delivered-item GMV, while 91.9% of eligible delivered orders arrived by their estimated delivery date. These figures indicate where demand and service exposure sit within Olist, not cross-border landed cost.
5. **Experience is associated with delivery outcomes.** Among reviewed delivered orders, late deliveries averaged 2.57/5 versus 4.29/5 for on-time deliveries; low-score shares were 54.1% and 9.2%, respectively. The first delivery-day group whose mean score fell below the overall reviewed-delivered-order mean was day 16, using only groups with at least 30 reviews and the first 99% of delivery durations. These are associations, not causal effects or a guaranteed service threshold.
6. **Market-entry evidence is exploratory.** The 52-category regression describes same-period Olist demand patterns; its 13-category holdout achieved R² = 0.845 and mean absolute error of 866 orders. This does not estimate entry success for a Chinese company. The category shortlist and KPI evidence identify hypotheses for external validation, especially import rules, landed costs, local competition, and current demand.

## 13. Business Recommendations

The actions below are testable priorities for a potential entrant, not claims of proven profitability or entry success.

| Priority | Evidence from Olist | Recommended next step | Pilot KPI | Main uncertainty |
|---|---|---|---|---|
| 1. Validate a narrow category set | 24.3% of categories generated 80% of platform GMV; Section 11.3 screens demand, freight, review, and seller concentration | Select a small set of shortlisted categories for current-market research and landed-cost calculations before sourcing inventory | Category contribution margin after duties, freight, taxes, returns, and platform fees | Historical platform demand may not represent current national demand or Chinese-sourced products |
| 2. Test focused regional distribution | Five states generated 73.9% of delivered-item GMV | Start with a limited regional pilot and compare service levels by state before expanding coverage | Delivered orders, contribution margin, on-time rate, and low-score rate by state | Population, advertising reach, competition, and local fulfillment costs are not in the dataset |
| 3. Design delivery and service safeguards | On-time rate was 91.9%; late reviewed deliveries had a 54.1% low-score share versus 9.2% on-time | Set realistic delivery promises, monitor carrier exceptions, and test proactive support for delayed orders | On-time rate, median delivery days, low-score rate, and returns per delivered order | The observed relationship is non-causal and domestic shipping is not cross-border shipping |
| 4. Measure repeat demand before scaling acquisition | The top 5% of spend-ranked buyers contributed 26.8% of spend; only 2.9% of 2017 consumers with defined payment records placed multiple observed orders | Pilot post-purchase retention and replenishment journeys where product fit supports them | Repeat purchase rate within a fixed follow-up window and net revenue per acquired consumer | Short customer histories and category mix may explain low observed repeat purchase |
| 5. Obtain external go/no-go evidence | Olist lacks import duties, compliance, local competitor pricing, product costs, and acquisition costs | Validate customs classification, regulatory requirements, current competitor offers, and customer willingness to pay for each pilot SKU | Fully landed unit margin, conversion rate, customer acquisition cost, and return rate | No entry decision should rely on Olist-only KPIs or the exploratory regression |

## 14. Analysis Limitations

- **Coverage and time.** The data represents historical Olist marketplace transactions, not all Brazilian e-commerce. The first and last calendar years are partial; 2017 is the only complete calendar year. Platform category and seller shares are not national shares.
- **Financial scope.** GMV is delivered-item price before freight and is not net revenue or profit. Product cost, platform fees, advertising, inventory, taxes, returns, customs duties, and cross-border landed costs are unavailable; no margin or investment-return claim is supported.
- **Customer observation.** Anonymized records lack demographics, income, acquisition channel, and registration date. Repeat purchase is measured only within the observation window and can understate longer-term retention. The 2017 active-month and active-year spend means have different denominators.
- **Reviews and delivery.** Not every order has a review. Reviews are reduced to the latest answered record per order for score analysis, and missing comment text is not a missing score. The day-16 threshold uses daily bins with 30 or more reviews and excludes the longest 1% of delivery durations; day-level fluctuations and selection bias remain. Delivery-review differences are associative, not causal.
- **Geography and cartography.** State comparison is not population-adjusted. The offline Natural Earth outline is simplified, and state map points are median geolocations rather than state polygons. State review colors use standardized differences between state mean scores; they are not significance tests, and low-volume states are less precise.
- **Model uncertainty.** The three-month historical forecast trains on 20 months ending August 2018; its 95% intervals cover the expected mean under model assumptions, not individual future outcomes or structural shocks. The 52-category entry regression uses same-period predictors; random category holdout fit does not establish future validity, causal effects, or market-entry success probability.
- **External validation.** Current Brazil-wide demand, competitor economics, import compliance, customer willingness to pay, and cross-border fulfillment are outside these files and must be checked before any real entry decision.

## 15. Conclusion

Historical Olist transactions show a sizable but concentrated platform business: R$13.22 million in delivered-item GMV, 96,478 delivered orders, substantial concentration across categories and customer states, and meaningful variation in freight and review outcomes. Low observed repeat purchasing and the strong association between late delivery and poor ratings make customer experience and retention important pilot questions.

For a potential China-to-Brazil entrant, these results identify **where to investigate**, not whether entry will succeed. The next decision should be a limited category-and-region pilot only after current demand, landed unit economics, regulatory fit, competitive offers, and service feasibility have been independently verified. Pilot results should then be judged on contribution margin, conversion, repeat purchasing, delivery reliability, and customer ratings rather than on the exploratory regression score.

## 16. Appendix

### 16.1 Metric Definitions

| Metric | Definition and population |
|---|---|
| Delivered-item GMV | Sum of `price` in order items whose `order_id` has status `delivered`; excludes charged freight. |
| Average order value | Delivered-item GMV divided by the number of distinct delivered orders. |
| Active customer | Distinct `customer_unique_id` attached to a delivered order. |
| Active seller | Distinct `seller_id` attached to a delivered order item. |
| Category GMV share | Category delivered-item GMV divided by total categorized delivered-item GMV within Olist. |
| Seller HHI by category | Sum of squared seller GMV shares within each Olist category; 1 indicates one seller holds all observed category GMV. |
| Charged freight / item GMV | Sum of `freight_value` divided by sum of `price` on delivered order items; not a carrier-cost or international-shipping measure. |
| On-time delivery rate | Share of delivered orders with a non-missing purchase-to-delivery duration and `order_delivered_customer_date` no later than `order_estimated_delivery_date`. |
| Low- and high-score review rates | Shares of reviewed orders with scores 1–2 and 4–5, respectively. Review analysis keeps at most the latest answered review per `order_id`. |
| State review standard score | (State mean review score − national reviewed-delivered-order mean) / review-count-weighted standard deviation of state means. Color is capped at ±2 SD; tooltip retains the underlying score. |

### 16.2 Reproducibility and Scope

The notebook reads nine Olist CSV files from `source_data/` and a local Natural Earth Brazil outline from `source_data/brazil_boundary.geojson`. Orders join to customers through `customer_id`, order items and payments through `order_id`, and products through `product_id`; seller analysis uses `seller_id`. Category labels are translated with `product_category_name_translation.csv`. Section 4 shows file-level missingness and duplicates before feature construction. Rerun the notebook from the first cell so all downstream tables and figures use the same definitions.

The 2017 spend metrics use defined payment records and distinct `customer_unique_id` within active months or the active year. The delivery-day threshold in Section 9.1 requires at least 30 reviewed orders per integer day and uses the first 99% of nonnegative delivery durations. The entry-model sample requires at least 100 delivered orders and 30 reviewed orders per category; its holdout split is seeded with 42.